# 18 · Top 100 LeetCode questions — part 4: heaps, backtracking and graphs

After this notebook you can solve the 20 most-asked **easy and medium** questions about *"the k biggest"*,
*"list every way"* and *"islands, maps and networks"* — and, more importantly, recognise which of three tricks a
brand-new question wants: a **heap** (a pile that always hands you the smallest), **backtracking** (try a choice,
explore, undo it) or a **graph walk** (BFS ripples or a DFS dive). Every answer is plain Python 3, checked with
`assert`s, and comes with an animation you can step through.

**Time:** ~60 min · **Runs:** offline on CPU in seconds · **Needs:** [08 · Recursion and backtracking](08_recursion_and_backtracking.ipynb), [10 · Heaps and top-k](10_heaps_and_top_k.ipynb), [11 · Graphs: BFS, DFS, topo sort, Dijkstra](11_graphs_bfs_dfs_topo_dijkstra.ipynb), [14 · Tries, union-find and LRU cache](14_tries_union_find_lru_cache.ipynb)

## How to use this notebook

1. Read the 🧸 story and the question.
2. **Cover the code and try it yourself** — 10 minutes, even if you only get the idea in words.
3. Run the answer, then press **▶ play** on the animation and watch every step. Use **◀ back** whenever a step
   surprises you.
4. Tomorrow, solve it again from a blank cell. That is when it sticks.

**Colours** mean the same thing in every picture: yellow = looking at it now, green = part of the answer,
red = no / dead end, grey = finished, blue = inside the window (here: on my path, or reached by the water),
orange = moving, purple = just added.

## The 20 in this notebook

| # | Problem | LeetCode | Level | Trick |
|---|---|---|---|---|
| 61 | Kth Largest Element in an Array | [215](https://leetcode.com/problems/kth-largest-element-in-an-array/) | 🟡 Medium | min-heap of size k |
| 62 | K Closest Points to Origin | [973](https://leetcode.com/problems/k-closest-points-to-origin/) | 🟡 Medium | max-heap of size k |
| 63 | Task Scheduler | [621](https://leetcode.com/problems/task-scheduler/) | 🟡 Medium | heap + cool-down queue (or a formula) |
| 64 | Subsets | [78](https://leetcode.com/problems/subsets/) | 🟡 Medium | backtracking: choose, explore, undo |
| 65 | Permutations | [46](https://leetcode.com/problems/permutations/) | 🟡 Medium | backtracking with a "used" list |
| 66 | Combination Sum | [39](https://leetcode.com/problems/combination-sum/) | 🟡 Medium | backtracking + prune when too big |
| 67 | Letter Combinations of a Phone Number | [17](https://leetcode.com/problems/letter-combinations-of-a-phone-number/) | 🟡 Medium | backtracking, one level per digit |
| 68 | Generate Parentheses | [22](https://leetcode.com/problems/generate-parentheses/) | 🟡 Medium | backtracking with two counters |
| 69 | Word Search | [79](https://leetcode.com/problems/word-search/) | 🟡 Medium | DFS on a grid + mark and unmark |
| 70 | Number of Islands | [200](https://leetcode.com/problems/number-of-islands/) | 🟡 Medium | flood fill each new piece of land |
| 71 | Max Area of Island | [695](https://leetcode.com/problems/max-area-of-island/) | 🟡 Medium | flood fill and count |
| 72 | Flood Fill | [733](https://leetcode.com/problems/flood-fill/) | 🟢 Easy | BFS from one cell |
| 73 | Rotting Oranges | [994](https://leetcode.com/problems/rotting-oranges/) | 🟡 Medium | multi-source BFS, one ring per minute |
| 74 | Clone Graph | [133](https://leetcode.com/problems/clone-graph/) | 🟡 Medium | BFS + old→new dictionary |
| 75 | Course Schedule | [207](https://leetcode.com/problems/course-schedule/) | 🟡 Medium | topological sort (Kahn) = cycle check |
| 76 | Course Schedule II | [210](https://leetcode.com/problems/course-schedule-ii/) | 🟡 Medium | topological sort, return the order |
| 77 | Pacific Atlantic Water Flow | [417](https://leetcode.com/problems/pacific-atlantic-water-flow/) | 🟡 Medium | BFS backwards from both oceans |
| 78 | Number of Provinces | [547](https://leetcode.com/problems/number-of-provinces/) | 🟡 Medium | union-find (or DFS) |
| 79 | Implement Trie (Prefix Tree) | [208](https://leetcode.com/problems/implement-trie-prefix-tree/) | 🟡 Medium | trie: a tree of letters |
| 80 | Network Delay Time | [743](https://leetcode.com/problems/network-delay-time/) | 🟡 Medium | Dijkstra: heap of (time, node) |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), Path.cwd() / "03_dsa", *Path.cwd().parents]:
    if (_p / "dsa_anim.py").exists():
        sys.path.insert(0, str(_p)); break
from dsa_anim import Movie, arr, bars, hmap, stack, queue, chain, chain_of, tree, grid, graph, note, say


class _Box:
    '''A tiny tree node, used only to draw a heap list as the tree it really is.'''
    def __init__(self, val):
        self.val, self.left, self.right = val, None, None


def heap_tree(h, title="the same heap, drawn as a tree", hi=None):
    '''Draw a heap list as a tree: the children of box i are boxes 2i+1 and 2i+2. hi={index: role}.'''
    boxes = [_Box(v) for v in h]
    for i, b in enumerate(boxes):
        b.left = boxes[2 * i + 1] if 2 * i + 1 < len(boxes) else None
        b.right = boxes[2 * i + 2] if 2 * i + 2 < len(boxes) else None
    return tree(boxes[0] if boxes else None, title, hi={boxes[i]: r for i, r in (hi or {}).items() if i < len(boxes)})


def lists_panel(items, title="answers found", hi=None):
    '''A row of answer cards: each list is written like [1,2], each string as it is.'''
    return arr(["[" + ",".join(map(str, x)) + "]" if isinstance(x, list) else x for x in items],
               title, hi=hi, index=False)

## The tricks in this notebook, in pictures

Four little movies, one per idea. Each problem below is one of these ideas wearing a costume.

1. **A heap** is a pile where the smallest thing always floats to the top. Python's `heapq` keeps it in a plain
   list; box `i` has its two children at `2i+1` and `2i+2`.
2. **Backtracking** is walking a maze: take a step, keep going, and at a dead end **go back one step (undo)** and
   try the next way.
3. **BFS** (breadth-first search) is ripples in a pond: a queue spreads the search one ring at a time, so the
   first time you reach a cell is the **shortest** way there.
4. **DFS** (depth-first search) dives down one path all the way, then comes back and tries the next one.

In [ ]:
import heapq
from collections import deque


def heap_pattern_movie(nums):
    m = Movie("A heap keeps the smallest on top", "heapq.heappush and heapq.heappop, drawn two ways")
    h, popped = [], []
    m.step("A **heap** is a pile where the smallest number is always on top, at index `0`. It starts empty.",
           arr(nums, "numbers to add"), arr(h, "heap (a plain list)"), heap_tree(h))
    for i, x in enumerate(nums):
        heapq.heappush(h, x)
        at = h.index(x)
        m.step(f"Push `{x}`: it goes in at the bottom, then swaps upward while it is smaller than its parent. "
               f"Top is now `{h[0]}`.",
               arr(nums, "numbers to add", hi={**{j: "done" for j in range(i)}, i: "look"}),
               arr(h, "heap (a plain list)", hi={at: "new"}), heap_tree(h, hi={at: "new"}))
    for _ in range(2):
        top = heapq.heappop(h)
        popped.append(top)
        m.step(f"Pop the top, `{top}` — always the smallest. The last box jumps to the top and sinks to its "
               f"place; `{h[0]}` is the new top.",
               arr(popped, "popped, in order", hi={len(popped) - 1: "good"}),
               arr(h, "heap (a plain list)", hi={0: "look"}), heap_tree(h, hi={0: "look"}))
    m.step("Push and pop each cost only **O(log n)** swaps (the height of the tree). Peeking at the top is "
           "**O(1)**. Every parent is smaller than its children.",
           arr(popped, "popped, in order", hi={0: "good", 1: "good"}), arr(h, "heap (a plain list)"),
           heap_tree(h))
    return popped, m


popped, m = heap_pattern_movie([5, 3, 8, 1, 6, 2])
assert popped == [1, 2]
m.show()

In [ ]:
def maze_movie(maze):
    m = Movie("Backtracking = walking a maze", "Try a way; at a dead end go back one step and try the next way")
    rows, cols = len(maze), len(maze[0])
    path, dead = [], set()

    def pic(caption, extra=None):
        hi = {cell: "bad" for cell in dead}
        hi.update({cell: "win" for cell in path})
        if path:
            hi[path[-1]] = "look"
        hi.update(extra or {})
        m.step(caption, grid(maze, "the maze  (█ = wall, S = start, E = exit)", hi=hi),
               arr([f"{r},{c}" for r, c in path], "my path (row,col)",
                   hi={i: "good" for i in range(len(path))} if extra else {len(path) - 1: "new"} if path else None))

    def walk(r, c):
        path.append((r, c))                                  # choose: step onto this cell
        if maze[r][c] == "E":
            pic("🎉 **The exit!** The path we kept is the way out. The red cells were dead ends we backed out of.",
                {cell: "good" for cell in path})
            return True
        pic(f"Step onto `{r},{c}`. From here, try right, then down, then left, then up.")
        for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
            nr, nc = r + dr, c + dc
            if (0 <= nr < rows and 0 <= nc < cols and maze[nr][nc] != "█"
                    and (nr, nc) not in path and (nr, nc) not in dead):
                if walk(nr, nc):                             # explore
                    return True
        path.pop()                                           # undo: this cell leads nowhere
        dead.add((r, c))
        back = f"`{path[-1][0]},{path[-1][1]}`" if path else "the start"
        pic(f"`{r},{c}` is a **dead end** — every way is a wall or already tried. **Go back one step (undo)** "
            f"to {back} and try its next way.")
        return False

    found = walk(0, 0)
    return (list(path) if found else None), m


maze = ["S..█",
        ".█.█",
        ".██.",
        "...E"]
way, m = maze_movie(maze)
assert way == [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2), (3, 3)]
m.show()

In [ ]:
def ripple_movie(rows, cols, stone):
    m = Movie("BFS = ripples in a pond", "A queue spreads the search one ring at a time")
    steps = [[None] * cols for _ in range(rows)]
    steps[stone[0]][stone[1]] = 0
    q, ring = deque([stone]), 0
    m.step("Drop a stone in the pond. The stone's cell is `0` steps away. The queue holds the cells of the "
           "newest ring.", grid(steps, "steps from the stone", hi={stone: "look"}),
           queue([f"{r},{c}" for r, c in q], "queue"))
    while q:
        ring_cells = []
        for _ in range(len(q)):                              # everything in the queue now is the same ring
            r, c = q.popleft()
            for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                if 0 <= nr < rows and 0 <= nc < cols and steps[nr][nc] is None:
                    steps[nr][nc] = ring + 1                 # first time the water gets here = fewest steps
                    q.append((nr, nc))
                    ring_cells.append((nr, nc))
        if not ring_cells:
            break
        ring += 1
        old = {(r, c): "done" for r in range(rows) for c in range(cols)
               if steps[r][c] is not None and steps[r][c] < ring}
        m.step(f"Ring {ring}: every empty cell touching ring {ring - 1} gets the number `{ring}`. "
               f"The ripple grows by exactly **one** step.",
               grid(steps, "steps from the stone", hi={**old, **{cell: "new" for cell in ring_cells}}),
               queue([f"{r},{c}" for r, c in q], "queue", hi={i: "new" for i in range(len(q))}))
    m.step("Every cell has a number. Going ring by ring makes each number the **shortest** way from the stone — "
           "that is why BFS answers fewest-steps questions.",
           grid(steps, "steps from the stone", hi={(r, c): "good" for r in range(rows) for c in range(cols)}))
    return steps, m


steps, m = ripple_movie(5, 5, (2, 2))
assert steps[0][0] == 4 and steps[2][2] == 0 and steps[1][2] == 1
m.show()

In [ ]:
def dfs_movie(adj, start, pos=None):
    m = Movie("DFS = dive down one path, then come back", "Recursion keeps the way back for us")
    seen, order, path, used = set(), [], [], {}
    names = list(adj)
    edges = sorted({tuple(sorted((u, v))) for u in adj for v in adj[u]})

    def pic(caption, cur=None, extra=None):
        hi = {u: "done" for u in seen}
        hi.update({u: "win" for u in path})
        if cur is not None:
            hi[cur] = "look"
        hi.update(extra or {})
        m.step(caption, graph(names, edges, "the map", hi=hi, ehi=used, pos=pos),
               stack(path, "the way back"), arr(order, "visit order", index=False))

    def go(u):
        seen.add(u); order.append(u); path.append(u)
        new = [v for v in adj[u] if v not in seen]
        pic(f"Visit **{u}**. " + (f"New friends: {', '.join(new)}. Dive into `{new[0]}` first."
                                  if new else "No new friends here."), u)
        for v in adj[u]:
            if v not in seen:
                used[(u, v)] = "win"
                go(v)                                         # dive all the way down this path
                left = [w for w in adj[u] if w not in seen]
                pic(f"**Back at {u}.** " + (f"`{left[0]}` is still new — dive there next." if left
                                            else f"Everything around {u} is visited."), u)
        path.pop()

    go(start)
    pic(f"Everything reachable from {start} is visited: {' → '.join(order)}. DFS went deep first and came "
        f"back only when stuck.", extra={u: "good" for u in order})
    return order, m


adj = {"A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"], "D": ["B"], "E": ["B", "F"], "F": ["C", "E"]}
pos = {"A": (0.5, 0), "B": (0.25, 0.5), "C": (0.75, 0.5), "D": (0.05, 1), "E": (0.45, 1), "F": (0.9, 1)}
order, m = dfs_movie(adj, "A", pos)
assert order == ["A", "B", "D", "E", "F", "C"]
m.show()

---

## 61 · Kth Largest Element in an Array · [LeetCode 215](https://leetcode.com/problems/kth-largest-element-in-an-array/) · 🟡 Medium

**🧸 Like you're five:** A club only has room for 3 of the biggest kids. Each new kid walks in; if the club gets too full,
        the smallest kid in the club has to leave. At the end, the smallest kid still in the club is the 3rd
        biggest of everyone.

**The question:** Given a list `nums` and a number `k`, return the **k-th largest** element (in sorted order, so duplicates count separately — not the k-th *distinct* one). Try to beat sorting.

| Input | Output |
|---|---|
| `nums = [3,2,1,5,6,4], k = 2` | `5` |
| `nums = [3,2,3,1,2,4,5,5,6], k = 4` | `4` |
| `nums = [1], k = 1` | `1` |

**🔎 How to spot it:** "**k-th largest / k biggest / top k**" → a **min-heap that never holds more than k** items. (The
        k-th *smallest* is the mirror: a max-heap of size k.)

**💡 The trick (min-heap of size k):** Walk the numbers and push each into a min-heap. When the heap has **k + 1** items, pop the smallest
        — it can't be in the top k. At the end the heap holds exactly the k biggest, and its top `heap[0]` (the
        smallest of them) is the answer. People often reach for a *max*-heap here; a min-heap of size k is the trick
        that keeps memory at O(k).

**⏱ Cost:** **O(n log k)** time (each push/pop costs log k) · **O(k)** space. Sorting is O(n log n). Quickselect is O(n) on average but O(n²) in the worst case.

In [ ]:
import heapq
import random


def find_kth_largest(nums, k):
    heap = []                                  # min-heap: the k biggest numbers seen so far
    for x in nums:
        heapq.heappush(heap, x)
        if len(heap) > k:                      # one too many: the smallest cannot be in the top k
            heapq.heappop(heap)
    return heap[0]                             # the smallest of the k biggest = the k-th biggest

assert find_kth_largest([3, 2, 1, 5, 6, 4], 2) == 5
assert find_kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4) == 4   # duplicates count separately
assert find_kth_largest([1], 1) == 1
assert find_kth_largest([-1, -5, -3], 3) == -5
rng = random.Random(0)
for _ in range(300):                           # compare with the slow, obvious answer
    nums = [rng.randint(-20, 20) for _ in range(rng.randint(1, 12))]
    k = rng.randint(1, len(nums))
    assert find_kth_largest(nums, k) == sorted(nums, reverse=True)[k - 1]
print(find_kth_largest([3, 2, 1, 5, 6, 4], 2))

In [ ]:
def kth_largest_movie(nums, k):
    m = Movie("Kth Largest Element", f"Keep a club of the {k} biggest; its smallest member is the answer")
    heap = []
    m.step(f"We want number **{k}** from the top. The club (a min-heap) has room for {k}; its smallest "
           f"member always sits on top.", arr(nums, "numbers"), arr(heap, "club = min-heap"), heap_tree(heap))
    for i, x in enumerate(nums):
        past = {j: "done" for j in range(i)}
        heapq.heappush(heap, x)
        at = heap.index(x)
        if len(heap) <= k:
            m.step(f"`{x}` joins the club. There is still room, so nobody leaves. Smallest member: `{heap[0]}`.",
                   arr(nums, "numbers", hi={**past, i: "look"}, ptr={"i": i}),
                   arr(heap, "club = min-heap", hi={at: "new"}), heap_tree(heap, hi={at: "new"}))
            continue
        m.step(f"`{x}` joins — now the club has {k + 1} members, **one too many**. The smallest, `{heap[0]}`, "
               f"is on top.", arr(nums, "numbers", hi={**past, i: "look"}, ptr={"i": i}),
               arr(heap, "club = min-heap", hi={at: "new", 0: "bad"}), heap_tree(heap, hi={at: "new", 0: "bad"}))
        out = heapq.heappop(heap)
        why = "the newcomer itself was too small" if out == x else f"it can't be in the top {k}"
        m.step(f"Pop `{out}` out of the club — {why}. The new top is `{heap[0]}`.",
               arr(nums, "numbers", hi={**past, i: "done"}),
               arr(heap, "club = min-heap", hi={0: "look"}), heap_tree(heap, hi={0: "look"}))
    m.step(f"Done! The club holds the {k} biggest: {sorted(heap, reverse=True)}. Its smallest member, "
           f"**`{heap[0]}`**, is number {k} from the top.",
           arr(nums, "numbers", hi={j: ("good" if x == heap[0] else "done") for j, x in enumerate(nums)}),
           arr(heap, "club = min-heap", hi={0: "good"}), heap_tree(heap, hi={0: "good"}))
    return heap[0], m

nums, k = [4, 1, 7, 3, 8, 5, 2], 3
answer, m = kth_largest_movie(nums, k)
assert answer == find_kth_largest(nums, k) == 5
m.show()

---

## 62 · K Closest Points to Origin · [LeetCode 973](https://leetcode.com/problems/k-closest-points-to-origin/) · 🟡 Medium

**🧸 Like you're five:** Your toys are scattered on the floor around you. Mum says: keep the 2 closest toys, put the rest
        away. You keep a pile of 2; whenever a third one joins, the one **farthest** away goes back to the box.

**The question:** Given `points[i] = [x, y]` and `k`, return the `k` points closest to `(0, 0)` (straight-line distance). Any order is accepted; the answer is guaranteed to be unique.

| Input | Output |
|---|---|
| `points = [[1,3],[-2,2]], k = 1` | `[[-2,2]]` |
| `points = [[3,3],[5,-1],[-2,4]], k = 2` | `[[3,3],[-2,4]]` |
| `points = [[0,1]], k = 1` | `[[0,1]]` |

**🔎 How to spot it:** "the **k closest / k smallest**" → keep a **max-heap of size k**: the worst of your k sits on top,
        ready to be thrown out. Python only has a min-heap, so store the **negative** distance.

**💡 The trick (max-heap of size k):** 1. Distance to the origin is `x² + y²` — skip the square root; it doesn't change which is closer.
        2. Push `(-distance, x, y)`. The most negative = the farthest point sits on top.
        3. When the heap holds k + 1 points, pop — the farthest one leaves.
        4. What remains are the k closest. The common slip: using a min-heap of *all* n points (works, but O(n) space
        and O(n log n) time).

**⏱ Cost:** **O(n log k)** time · **O(k)** space. Sorting all points by distance is O(n log n) and one line: `sorted(points, key=...)[:k]` — say it, then offer the heap.

In [ ]:
import heapq
import random


def k_closest(points, k):
    heap = []                                  # max-heap of the k closest so far, as (-distance, x, y)
    for x, y in points:
        heapq.heappush(heap, (-(x * x + y * y), x, y))   # squared distance: no square root needed
        if len(heap) > k:
            heapq.heappop(heap)                # the farthest of the k + 1 goes back in the box
    return [[x, y] for _, x, y in heap]

assert sorted(k_closest([[1, 3], [-2, 2]], 1)) == [[-2, 2]]
assert sorted(k_closest([[3, 3], [5, -1], [-2, 4]], 2)) == [[-2, 4], [3, 3]]
assert k_closest([[0, 1]], 1) == [[0, 1]]
assert sorted(k_closest([[1, 1], [2, 2], [1, 1]], 2)) == [[1, 1], [1, 1]]   # duplicate points
rng = random.Random(0)
d = lambda p: p[0] ** 2 + p[1] ** 2
for _ in range(300):
    pts = [[rng.randint(-9, 9), rng.randint(-9, 9)] for _ in range(rng.randint(1, 10))]
    k = rng.randint(1, len(pts))
    got = k_closest(pts, k)                    # ties are possible here, so compare the distances
    assert sorted(map(d, got)) == sorted(map(d, pts))[:k]
print(k_closest([[3, 3], [5, -1], [-2, 4]], 2))

In [ ]:
def k_closest_movie(points, k, names):
    m = Movie("K Closest Points to Origin", f"Keep the {k} closest toys; the farthest one sits on top")
    label = {tuple(p): nm for p, nm in zip(points, names)}
    xs, ys = range(-3, 4), range(2, -3, -1)

    def floor(hi):
        cells = [[label.get((x, y), "◎" if (x, y) == (0, 0) else None) for x in xs] for y in ys]
        where = {label[(x, y)]: (2 - y, x + 3) for x, y in label}
        return grid(cells, "the floor (◎ = you, at 0,0)", hi={where[n]: r for n, r in hi.items()},
                    row_labels=list(ys), col_labels=list(xs))

    def pile(heap, hi=None):
        return arr([f"{label[(x, y)]}:{-nd}" for nd, x, y in heap], "max-heap  name:distance²  (top = farthest)",
                   hi=hi)

    heap, gone = [], []
    m.step(f"Toys A–E lie on the floor. Distance² = x² + y² (no square root needed). Keep the **{k}** closest.",
           floor({}), pile(heap), hmap({n: x * x + y * y for (x, y), n in zip(map(tuple, points), names)},
                                       "distance² of each toy"))
    for (x, y), nm in zip(points, names):
        heapq.heappush(heap, (-(x * x + y * y), x, y))
        kept = {label[(a, b)]: "win" for _, a, b in heap}
        if len(heap) <= k:
            m.step(f"Toy **{nm}** at ({x},{y}) is {x * x + y * y} away. The pile has room, keep it.",
                   floor({**{g: "done" for g in gone}, **kept, nm: "new"}), pile(heap))
            continue
        far = label[(heap[0][1], heap[0][2])]
        m.step(f"Toy **{nm}** is {x * x + y * y} away. Now {k + 1} toys in the pile: too many. The farthest, "
               f"**{far}** ({-heap[0][0]}), is on top.",
               floor({**{g: "done" for g in gone}, **kept, nm: "new", far: "bad"}), pile(heap, {0: "bad"}))
        heapq.heappop(heap)
        gone.append(far)
        m.step(f"Pop **{far}** — back to the toy box. The pile is {k} toys again.",
               floor({**{g: "done" for g in gone}, **{label[(a, b)]: "win" for _, a, b in heap}}), pile(heap))
    answer = [[x, y] for _, x, y in heap]
    m.step(f"Every toy has been seen. The {k} closest are " +
           " and ".join(f"**{label[(x, y)]}** ({x},{y})" for x, y in answer) + ".",
           floor({**{g: "done" for g in gone}, **{label[(x, y)]: "good" for x, y in answer}}),
           pile(heap, {i: "good" for i in range(len(heap))}))
    return answer, m

points = [[2, 2], [-1, 1], [3, -2], [0, -2], [-3, 0]]
answer, m = k_closest_movie(points, 2, "ABCDE")
assert sorted(answer) == sorted(k_closest(points, 2)) == [[-1, 1], [0, -2]]
m.show()

---

## 63 · Task Scheduler · [LeetCode 621](https://leetcode.com/problems/task-scheduler/) · 🟡 Medium

**🧸 Like you're five:** You have jobs to do: feed the cat 3 times, water the plants twice, read once. After you do a job,
        that same job needs a rest of 2 turns. Each turn you pick the job with the **most** left to do; if every job is
        resting, you wait one turn (idle).

**The question:** Given CPU `tasks` (letters) and a cool-down `n`, the same letter must be at least `n` slots apart. Each slot runs one task or is idle. Return the **minimum number of slots** to finish all tasks.

| Input | Output |
|---|---|
| `tasks = ["A","A","A","B","B","B"], n = 2` | `8` |
| `tasks = ["A","C","A","B","D","B"], n = 1` | `6` |
| `tasks = ["A","A","A","B","B","B"], n = 0` | `6` |

**🔎 How to spot it:** "**the same thing must wait n turns** before it can happen again" / "schedule to minimise time" →
        greedy: always run the task with the **most copies left** (a max-heap), and park the others in a waiting
        queue until their rest is over.

**💡 The trick (heap + cool-down queue):** Keep a max-heap of "how many left" per task and a queue of resting tasks `(left, task, ready_time)`.
        Each slot: pop the task with the most left and run it; if it still has copies, it rests in the queue for n
        slots. If the heap is empty, the slots are **idle** until the first resting task wakes, so jump the clock straight there. When the front of the queue is rested, push it back into
        the heap. Why most-left first? The biggest pile is the one that forces idles, so start it early.
        **Shortcut formula** (say it once you have the heap version): with `top` = the biggest count and `ties` = how
        many tasks have that count, the answer is `max(len(tasks), (top - 1) * (n + 1) + ties)`.

**⏱ Cost:** **O(len(tasks) · log 26)** ≈ O(len(tasks)) time — each run is one heap step, and a stretch of idle slots is skipped in one jump · **O(26)** space. The formula is **O(len(tasks))** time, O(1) space. Trying every order is factorial.

In [ ]:
import heapq
import random
from collections import Counter, deque


def least_interval(tasks, n):
    heap = [(-c, t) for t, c in Counter(tasks).items()]   # most copies left on top (negative = max-heap)
    heapq.heapify(heap)
    resting = deque()                          # (copies left as a negative, task, slot when rest ends)
    time = 0
    while heap or resting:
        time += 1                              # a new slot
        if heap:
            left, t = heapq.heappop(heap)      # run the task with the most copies left
            if left + 1 < 0:                   # copies remain: it must rest n slots
                resting.append((left + 1, t, time + n))
        else:                                  # every task is resting: idle until the first one wakes
            time = resting[0][2]               # jump over the idle slots in one go
        if resting and resting[0][2] == time:  # its rest is over: back to the ready pile
            left, t, _ = resting.popleft()
            heapq.heappush(heap, (left, t))
    return time


def least_interval_formula(tasks, n):
    counts = Counter(tasks).values()
    top = max(counts)
    ties = sum(1 for c in counts if c == top)
    return max(len(tasks), (top - 1) * (n + 1) + ties)   # frames of n+1 slots, plus the last row

assert least_interval(list("AAABBB"), 2) == 8           # A B idle A B idle A B
assert least_interval(["A"] * 10_000, 100) == 1_009_900  # LeetCode's biggest case: fast, idles are jumped
assert least_interval(list("ACABDB"), 1) == 6
assert least_interval(list("AAABBB"), 0) == 6           # no rest needed
assert least_interval(list("AAAAAABCDEFG"), 2) == 16
assert least_interval(["A"], 5) == 1
rng = random.Random(0)
for _ in range(300):
    tasks = [rng.choice("ABCD") for _ in range(rng.randint(1, 12))]
    n = rng.randint(0, 4)
    assert least_interval(tasks, n) == least_interval_formula(tasks, n)
print(least_interval(list("AAABBB"), 2))

In [ ]:
def task_scheduler_movie(tasks, n):
    m = Movie("Task Scheduler", f"Same job must rest {n} slots; run the job with the most left")
    heap = [(-c, t) for t, c in Counter(tasks).items()]
    heapq.heapify(heap)
    resting, timeline, time = deque(), [], 0

    def pic(caption, hi_ready=None):
        ready = {t: -c for c, t in sorted(heap)}
        m.step(caption, arr(timeline, "slots so far", hi={
                   **{i: ("bad" if s == "idle" else "done") for i, s in enumerate(timeline)},
                   **({len(timeline) - 1: "look"} if timeline and timeline[-1] != "idle" else {})}),
               hmap(ready, "ready pile (job → copies left)", hi=hi_ready),
               queue([f"{t}:{ready_at + 1}" for _, t, ready_at in resting], "resting (job:free at slot)"),
               note(slot=time))

    pic(f"Jobs: {' '.join(tasks)}. Run the job with the most copies left first — the biggest pile is the one "
        f"that could force us to wait.")
    while heap or resting:
        time += 1
        if heap:
            left, t = heapq.heappop(heap)
            timeline.append(t)
            if left + 1 < 0:
                resting.append((left + 1, t, time + n))
            msg = (f"Slot {time}: run **{t}** (it had the most left, {-left}). "
                   + (f"It must rest until slot {time + n + 1}." if left + 1 < 0 else f"That was its last copy."))
        else:
            timeline.append("idle")
            msg = f"Slot {time}: every job is resting, so we **wait** — an idle slot."
        back = None
        if resting and resting[0][2] == time:
            left, t2, _ = resting.popleft()
            heapq.heappush(heap, (left, t2))
            back = t2
            msg += f" **{t2}** finished resting — back to the ready pile."
        pic(msg, {back: "new"} if back else None)
    m.step(f"All jobs done in **{time}** slots: {' '.join(timeline)}. The idle slots are the red ones.",
           arr(timeline, "all slots", hi={i: ("bad" if s == "idle" else "good") for i, s in enumerate(timeline)}),
           note(answer=time))
    return time, m

tasks, n = list("AAABBC"), 2
answer, m = task_scheduler_movie(tasks, n)
assert answer == least_interval(tasks, n) == least_interval_formula(tasks, n) == 7
m.show()

---

## 64 · Subsets · [LeetCode 78](https://leetcode.com/problems/subsets/) · 🟡 Medium

**🧸 Like you're five:** You have 3 different sweets. How many different bags can you pack — including the empty bag and the
        bag with everything? Put a sweet in, pack the rest, then **take it back out** and try without it.

**The question:** Given `nums` of **distinct** integers, return all possible subsets (the power set). No duplicate subsets; any order.

| Input | Output |
|---|---|
| `nums = [1,2,3]` | `[[],[1],[1,2],[1,2,3],[1,3],[2],[2,3],[3]]` |
| `nums = [0]` | `[[],[0]]` |
| `nums = []` | `[[]]` |

**🔎 How to spot it:** "return **all** subsets / all combinations / every way" → backtracking. The answer list itself is
        exponential (2ⁿ), so no faster trick exists — the job is to list them cleanly.

**💡 The trick (backtracking):** Keep one `path` list. `go(start)`: first write down a copy of `path` (every path is a subset). Then for
        each index `i` from `start` on: **choose** `nums[i]` (append), **explore** `go(i + 1)`, **undo** (pop). Starting
        from `i + 1` means you only look to the right, so `[1,2]` and `[2,1]` are never both made. The classic bug:
        `ans.append(path)` instead of `path[:]` — you save the same list 8 times and it ends up empty.

**⏱ Cost:** **O(n · 2ⁿ)** time (2ⁿ subsets, copying each is up to n) · **O(n)** extra space for the path and recursion. A bitmask loop over `range(2**n)` is the same cost.

In [ ]:
from itertools import combinations


def subsets(nums):
    ans, path = [], []

    def go(start):
        ans.append(path[:])                    # every path is a subset: save a COPY
        for i in range(start, len(nums)):
            path.append(nums[i])               # choose
            go(i + 1)                          # explore (only look to the right)
            path.pop()                         # undo

    go(0)
    return ans

canon = lambda xss: sorted(sorted(x) for x in xss)
assert canon(subsets([1, 2, 3])) == canon([[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]])
assert canon(subsets([0])) == [[], [0]]
assert subsets([]) == [[]]
for n in range(7):                             # compare with itertools on 0..6 items
    nums = list(range(10, 10 + n))
    want = [list(c) for r in range(n + 1) for c in combinations(nums, r)]
    assert canon(subsets(nums)) == canon(want) and len(subsets(nums)) == 2 ** n
print(subsets([1, 2, 3]))

In [ ]:
def subsets_movie(nums):
    m = Movie("Subsets", "Choose, explore, undo — every path is a subset")
    ans, path, picked = [], [], []

    def pic(caption, new_answer=False):
        m.step(caption, arr(nums, "sweets", hi={i: "win" for i in picked}),
               arr(path, "path (my bag right now)", hi={len(path) - 1: "new"} if path else None),
               lists_panel(ans, f"subsets found: {len(ans)}", hi={len(ans) - 1: "good"} if new_answer else None))

    def go(start):
        ans.append(path[:])
        if not path:
            pic("Start with an empty bag. The empty bag `[]` is a subset too — write it down.", True)
        else:
            pic(f"**Choose** `{path[-1]}` → bag is {path}. That is a new subset: write it down.", True)
        for i in range(start, len(nums)):
            path.append(nums[i]); picked.append(i)
            go(i + 1)
            path.pop(); picked.pop()
            pic(f"**Go back one step (undo)**: take `{nums[i]}` out → bag is {path}. "
                + ("Try the next sweet to the right." if i + 1 < len(nums) else "No sweets left to the right."))

    go(0)
    m.step(f"All **{len(ans)}** subsets found (2 × 2 × 2 = 8: each sweet is either in or out).",
           arr(nums, "sweets"), lists_panel(ans, f"subsets found: {len(ans)}", hi={i: "good" for i in range(len(ans))}))
    return ans, m

nums = [1, 2, 3]
answer, m = subsets_movie(nums)
assert answer == subsets(nums)
m.show()

---

## 65 · Permutations · [LeetCode 46](https://leetcode.com/problems/permutations/) · 🟡 Medium

**🧸 Like you're five:** Three kids, one line. How many ways can they stand? Pick who stands first, then who stands second
        from the kids **still sitting**, and so on. When you have tried a kid in a spot, they sit back down (undo) and
        the next kid gets a turn.

**The question:** Given an array `nums` of **distinct** integers, return all possible permutations (orderings), in any order.

| Input | Output |
|---|---|
| `nums = [1,2,3]` | `[[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]]` |
| `nums = [0,1]` | `[[0,1],[1,0]]` |
| `nums = [1]` | `[[1]]` |

**🔎 How to spot it:** "all **orderings / arrangements**" (order matters, everyone used once) → backtracking with a
        **used** list. Compare: *subsets* only look to the right; *permutations* look at **everyone not used yet**.

**💡 The trick (backtracking with a used list):** `go()`: if `path` has all n numbers, save a copy. Otherwise loop over **every** index; skip the used
        ones; choose (mark used, append), explore, undo (pop, unmark). Forgetting to unmark `used` is the usual bug:
        the second branch then thinks nobody is left.

**⏱ Cost:** **O(n · n!)** time (n! orderings, each copied in n) · **O(n)** extra space. There is no faster way, since the output itself has n! lists.

In [ ]:
from itertools import permutations


def permute(nums):
    ans, path, used = [], [], [False] * len(nums)

    def go():
        if len(path) == len(nums):             # everyone is in line: one full ordering
            ans.append(path[:])
            return
        for i, x in enumerate(nums):
            if used[i]:
                continue
            used[i] = True; path.append(x)     # choose
            go()                               # explore
            path.pop(); used[i] = False        # undo — both halves!

    go()
    return ans

assert permute([1, 2, 3]) == [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
assert sorted(permute([0, 1])) == [[0, 1], [1, 0]]
assert permute([1]) == [[1]]
for n in range(1, 7):
    nums = list(range(-2, n - 2))              # negatives too
    assert sorted(permute(nums)) == sorted(map(list, permutations(nums)))
print(permute([1, 2, 3]))

In [ ]:
def permute_movie(nums):
    m = Movie("Permutations", "Pick a kid who is still sitting; when done, they sit back down")
    ans, path, used = [], [], [False] * len(nums)

    def pic(caption, new_answer=False):
        m.step(caption, arr(nums, "kids (blue = standing in line)", hi={i: "win" for i in range(len(nums)) if used[i]}),
               arr(path, "path (the line so far)", hi={len(path) - 1: "new"} if path else None),
               lists_panel(ans, f"orderings found: {len(ans)}", hi={len(ans) - 1: "good"} if new_answer else None))

    def go():
        if len(path) == len(nums):
            ans.append(path[:])
            pic(f"**Choose** `{path[-1]}` → the line {path} is full. A new ordering! Write it down.", True)
            return
        if path:
            pic(f"**Choose** `{path[-1]}` → line is {path}. Pick the next kid from those still sitting.")
        for i, x in enumerate(nums):
            if used[i]:
                continue
            used[i] = True; path.append(x)
            go()
            path.pop(); used[i] = False
            pic(f"**Go back one step (undo)**: `{x}` sits back down → line is {path}.")

    pic(f"Kids {nums} are all sitting. The line is empty. Try each kid in the first spot.")
    go()
    m.step(f"All **{len(ans)}** orderings found (3 × 2 × 1 = 6).", arr(nums, "kids"),
           lists_panel(ans, f"orderings found: {len(ans)}", hi={i: "good" for i in range(len(ans))}))
    return ans, m

nums = [1, 2, 3]
answer, m = permute_movie(nums)
assert answer == permute(nums)
m.show()

---

## 66 · Combination Sum · [LeetCode 39](https://leetcode.com/problems/combination-sum/) · 🟡 Medium

**🧸 Like you're five:** You have coins worth 2, 3, 6 and 7, as many of each as you like. Which piles make exactly 7? Add a
        coin; if the pile goes over, that way is hopeless — take the coin back and try another.

**The question:** Given **distinct** positive `candidates` and a `target`, return all **unique combinations** that sum to `target`. The same number may be used **any number of times**. Two combinations differ if the count of some number differs. Any order.

| Input | Output |
|---|---|
| `candidates = [2,3,6,7], target = 7` | `[[2,2,3],[7]]` |
| `candidates = [2,3,5], target = 8` | `[[2,2,2,2],[2,3,3],[3,5]]` |
| `candidates = [2], target = 1` | `[]` |

**🔎 How to spot it:** "all combinations that **add up to** a target" + "may reuse" → backtracking that passes `i` (not
        `i + 1`) to the next call. With positive numbers, **sort** first and stop as soon as a number is too big.

**💡 The trick (backtracking with pruning):** `go(start, left)`: if `left == 0`, save a copy. Otherwise for each `i` from `start`: if
        `candidates[i] > left`, **break** (sorted, so all later ones are too big — this is the pruning). Else choose
        it, call `go(i, left - c)` — `i` again, because the same coin may repeat — and undo. Starting at `start`
        (never going back left) is what stops `[2,3,2]` and `[3,2,2]` from both appearing.

**⏱ Cost:** **Exponential** time — roughly O(n^(target/min)) in the worst case, but pruning cuts most of it · **O(target/min)** space for the path. There is no polynomial way to *list* them all.

In [ ]:
import random


def combination_sum(candidates, target):
    cands = sorted(candidates)                 # sorted, so we can stop early
    ans, path = [], []

    def go(start, left):
        if left == 0:                          # exactly on target
            ans.append(path[:])
            return
        for i in range(start, len(cands)):
            c = cands[i]
            if c > left:                       # too big, and so is every coin after it: prune
                break
            path.append(c)                     # choose
            go(i, left - c)                    # explore — i, not i + 1: the same coin may repeat
            path.pop()                         # undo

    go(0, target)
    return ans


def count_ways(candidates, target):            # coin-change counting, used only to check the answer
    ways = [1] + [0] * target
    for c in candidates:
        for s in range(c, target + 1):
            ways[s] += ways[s - c]
    return ways[target]

canon = lambda xss: sorted(sorted(x) for x in xss)
assert canon(combination_sum([2, 3, 6, 7], 7)) == [[2, 2, 3], [7]]
assert canon(combination_sum([2, 3, 5], 8)) == [[2, 2, 2, 2], [2, 3, 3], [3, 5]]
assert combination_sum([2], 1) == []
assert combination_sum([1], 1) == [[1]]
rng = random.Random(0)
for _ in range(200):
    cands = rng.sample(range(1, 12), rng.randint(1, 4))
    target = rng.randint(1, 18)
    got = combination_sum(cands, target)
    assert all(sum(x) == target for x in got)                 # every pile is right
    assert len({tuple(sorted(x)) for x in got}) == len(got)   # no pile twice
    assert len(got) == count_ways(cands, target)              # and none is missing
print(combination_sum([2, 3, 6, 7], 7))

In [ ]:
def combination_sum_movie(candidates, target):
    m = Movie("Combination Sum", f"Coins may repeat; the pile must be exactly {target}")
    cands = sorted(candidates)
    ans, path = [], []

    def pic(caption, hi_coins=None, new_answer=False):
        m.step(caption, arr(cands, "coins (sorted)", hi=hi_coins),
               arr(path, "path (my pile)", hi={len(path) - 1: "new"} if path else None),
               note(still_need=target - sum(path)),
               lists_panel(ans, "piles found", hi={len(ans) - 1: "good"} if new_answer else None))

    def go(start, left):
        if left == 0:
            ans.append(path[:])
            pic(f"**Choose** `{path[-1]}` → {path} is exactly {target}! Write it down. 🎉", new_answer=True)
            return
        if path:
            pic(f"**Choose** `{path[-1]}` → pile {path}, still need `{left}`. Next coins: `{cands[start]}` or bigger.",
                {start: "look"})
        for i in range(start, len(cands)):
            c = cands[i]
            if c > left:
                pic(f"`{c}` is more than the `{left}` we need, and every coin after it is bigger (sorted). "
                    f"**Prune**: stop this branch.", {j: "bad" for j in range(i, len(cands))})
                break
            path.append(c)
            go(i, left - c)
            path.pop()
            pic(f"**Go back one step (undo)**: take `{c}` off → pile {path}.")

    pic(f"Coins {cands}, target {target}. Add coins from left to right; a coin may be used again.")
    go(0, target)
    m.step(f"Done: **{len(ans)}** piles make {target}. Every pile is in non-decreasing order, so no pile appears twice.",
           arr(cands, "coins (sorted)"), lists_panel(ans, "piles found", hi={i: "good" for i in range(len(ans))}))
    return ans, m

cands, target = [2, 3, 6, 7], 7
answer, m = combination_sum_movie(cands, target)
assert answer == combination_sum(cands, target) == [[2, 2, 3], [7]]
m.show()

---

## 67 · Letter Combinations of a Phone Number · [LeetCode 17](https://leetcode.com/problems/letter-combinations-of-a-phone-number/) · 🟡 Medium

**🧸 Like you're five:** On an old phone, key 2 has the letters a b c and key 3 has d e f. You press 2 then 3. Every word
        you could mean is one letter from key 2 followed by one letter from key 3: ad, ae, af, bd, …

**The question:** Given a string `digits` (each `2`–`9`), return all letter combinations the number could represent, using the phone keypad. Return `[]` for an empty string. Any order.

| Input | Output |
|---|---|
| `digits = "23"` | `["ad","ae","af","bd","be","bf","cd","ce","cf"]` |
| `digits = ""` | `[]` |
| `digits = "2"` | `["a","b","c"]` |

**🔎 How to spot it:** "all combinations, **one choice from each group**" (a fixed number of slots, each with its own
        options) → backtracking where **level i = position i**. Same shape as `itertools.product`.

**💡 The trick (backtracking, one level per digit):** `go(i)`: if `i == len(digits)`, save `"".join(path)`. Otherwise for each letter on key `digits[i]`:
        append it, `go(i + 1)`, pop. The trap: `""` must return `[]`, not `[""]` — check it first.

**⏱ Cost:** **O(4ⁿ · n)** time (up to 4 letters per key, each word is n long) · **O(n)** extra space. That is also the size of the output, so it can't be beaten.

In [ ]:
from itertools import product

PHONE = {"2": "abc", "3": "def", "4": "ghi", "5": "jkl", "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz"}


def letter_combinations(digits):
    if not digits:                             # the edge case: no digits means no words at all
        return []
    ans, path = [], []

    def go(i):
        if i == len(digits):                   # one letter for every digit
            ans.append("".join(path))
            return
        for ch in PHONE[digits[i]]:
            path.append(ch)                    # choose
            go(i + 1)                          # explore the next digit
            path.pop()                         # undo

    go(0)
    return ans

assert letter_combinations("23") == ["ad", "ae", "af", "bd", "be", "bf", "cd", "ce", "cf"]
assert letter_combinations("") == []
assert letter_combinations("2") == ["a", "b", "c"]
assert len(letter_combinations("79")) == 16          # 7 and 9 have four letters each
for digits in ["7", "234", "9999", "58"]:
    assert letter_combinations(digits) == ["".join(p) for p in product(*(PHONE[d] for d in digits))]
print(letter_combinations("23"))

In [ ]:
def letters_movie(digits):
    m = Movie("Letter Combinations", f"Pick one letter from each key: {' then '.join(digits)}")
    ans, path = [], []

    def pic(caption, i, letter=None, new_answer=False):
        m.step(caption,
               arr(list(digits), "keys pressed", hi={j: ("done" if j < i else "look") for j in range(min(i + 1, len(digits)))},
                   ptr={"i": i} if i < len(digits) else None),
               hmap({d: PHONE[d] for d in digits}, "keypad"),
               arr(path, "path", hi={len(path) - 1: "new"} if path else None),
               lists_panel(ans, f"words found: {len(ans)}", hi={len(ans) - 1: "good"} if new_answer else None))

    def go(i):
        if i == len(digits):
            ans.append("".join(path))
            pic(f"**Choose** `{path[-1]}` → `{''.join(path)}` has a letter for every key. Write it down!", i, new_answer=True)
            return
        if path:
            pic(f"**Choose** `{path[-1]}` → path `{''.join(path)}`. Now try each letter of key `{digits[i]}`.", i)
        for ch in PHONE[digits[i]]:
            path.append(ch)
            go(i + 1)
            path.pop()
            pic(f"**Go back one step (undo)**: remove `{ch}` → path `{''.join(path) or '(empty)'}`.", i)

    pic(f"Keys {', '.join(digits)}. Level 1 picks a letter of key `{digits[0]}`, level 2 a letter of key "
        f"`{digits[1] if len(digits) > 1 else digits[0]}`.", 0)
    go(0)
    m.step(f"Done: {' × '.join(str(len(PHONE[d])) for d in digits)} = **{len(ans)}** words.",
           arr(list(digits), "keys pressed"), lists_panel(ans, f"words found: {len(ans)}", hi={j: "good" for j in range(len(ans))}))
    return ans, m

answer, m = letters_movie("23")
assert answer == letter_combinations("23")
m.show()

---

## 68 · Generate Parentheses · [LeetCode 22](https://leetcode.com/problems/generate-parentheses/) · 🟡 Medium

**🧸 Like you're five:** You have 2 left hands `(` and 2 right hands `)`. Every right hand must hold a left hand that came
        before it. Add hands one at a time; you may add a right hand only if some left hand is still waiting for a
        partner.

**The question:** Given `n`, return all combinations of `n` pairs of parentheses that are **well-formed** (every `(` is closed, and never close more than you opened). Any order.

| Input | Output |
|---|---|
| `n = 3` | `["((()))","(()())","(())()","()(())","()()()"]` |
| `n = 1` | `["()"]` |
| `n = 2` | `["(())","()()"]` |

**🔎 How to spot it:** "generate all **valid** strings / arrangements" with a **rule you can check as you go** → build
        them one character at a time and only take steps that keep the rule. Don't make all 2²ⁿ strings and filter.

**💡 The trick (backtracking with two counters):** Carry two counters, `opened` and `closed`. You may add `(` while `opened < n`. You may add `)` only
        while `closed < opened` (someone is waiting to be closed). When the string has `2n` characters, it is valid
        by construction — save it. The common mistake: allowing `)` whenever `closed < n`, which makes `)(`.

**⏱ Cost:** **O(4ⁿ / √n)** time — that is the n-th Catalan number of answers, times n to build each · **O(n)** space for the path. Brute force (all 2²ⁿ strings, then check each) is O(2²ⁿ · n).

In [ ]:
from itertools import product


def generate_parenthesis(n):
    ans, path = [], []

    def go(opened, closed):
        if len(path) == 2 * n:                 # full length, valid by construction
            ans.append("".join(path))
            return
        if opened < n:                         # a new "(" is always fine while we have some left
            path.append("("); go(opened + 1, closed); path.pop()
        if closed < opened:                    # ")" only if some "(" is still waiting
            path.append(")"); go(opened, closed + 1); path.pop()

    go(0, 0)
    return ans


def is_valid(s):
    depth = 0
    for ch in s:
        depth += 1 if ch == "(" else -1
        if depth < 0:
            return False
    return depth == 0

assert generate_parenthesis(3) == ["((()))", "(()())", "(())()", "()(())", "()()()"]
assert generate_parenthesis(1) == ["()"]
assert generate_parenthesis(2) == ["(())", "()()"]
for n in range(1, 7):                          # brute force: every string of ( and ), keep the valid ones
    brute = sorted(s for s in map("".join, product("()", repeat=2 * n)) if is_valid(s))
    assert sorted(generate_parenthesis(n)) == brute
print(generate_parenthesis(3))

In [ ]:
def parens_movie(n):
    m = Movie("Generate Parentheses", f"{n} pairs: open while you can, close only what is open")
    ans, path = [], []

    def pic(caption, opened, closed, new_answer=False, bad=False):
        m.step(caption, arr(path, "path", hi={len(path) - 1: ("bad" if bad else "new")} if path else None),
               note(opened=opened, closed=closed, n=n),
               lists_panel(ans, "valid strings found", hi={len(ans) - 1: "good"} if new_answer else None))

    def go(opened, closed):
        if len(path) == 2 * n:
            ans.append("".join(path))
            pic(f"**Choose** `{path[-1]}` → `{''.join(path)}` is full length and valid. Write it down!",
                opened, closed, new_answer=True)
            return
        if path:
            pic(f"**Choose** `{path[-1]}` → `{''.join(path)}`.", opened, closed)
        if opened < n:
            path.append("("); go(opened + 1, closed); path.pop()
            pic(f"**Go back one step (undo)**: remove `(` → `{''.join(path) or '(empty)'}`.", opened, closed)
        else:
            pic(f"Can't add `(`: all {n} are used. Try `)` instead.", opened, closed)
        if closed < opened:
            path.append(")"); go(opened, closed + 1); path.pop()
            pic(f"**Go back one step (undo)**: remove `)` → `{''.join(path) or '(empty)'}`.", opened, closed)
        else:
            pic(f"Can't add `)`: {'nothing is open' if opened == 0 else 'every ( is already closed'}. "
                f"That way would break the rule.", opened, closed)

    pic(f"Build strings of length {2 * n}. Rule 1: at most {n} `(`. Rule 2: a `)` needs an open `(` waiting.", 0, 0)
    go(0, 0)
    m.step(f"Done: **{len(ans)}** valid strings, and we never built a broken one.",
           lists_panel(ans, "valid strings found", hi={i: "good" for i in range(len(ans))}))
    return ans, m

answer, m = parens_movie(2)
assert answer == generate_parenthesis(2) == ["(())", "()()"]
m.show()

---

## 69 · Word Search · [LeetCode 79](https://leetcode.com/problems/word-search/) · 🟡 Medium

**🧸 Like you're five:** A letter grid is on the table. Can you trace the word with your finger, moving only to the
        square left, right, up or down, and never touching the same square twice? If your finger gets stuck, lift it
        back one square and try another direction.

**The question:** Given an `m × n` grid of letters `board` and a string `word`, return `True` if `word` can be built from letters of **adjacent** cells (up/down/left/right). The same cell may not be used more than once.

| Input | Output |
|---|---|
| `board = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]], word = "ABCCED"` | `True` |
| `same board, word = "SEE"` | `True` |
| `same board, word = "ABCB"` | `False` |

**🔎 How to spot it:** "find a **path in a grid** that spells / matches something, **each cell once**" → DFS from every
        cell, with **mark → explore → unmark** (backtracking on a grid).

**💡 The trick (grid dfs + backtracking):** Try every cell as the start. `dfs(r, c, i)`: fail if off the board or `board[r][c] != word[i]`.
        Otherwise **mark** the cell (write `#`, so the path can't reuse it), try the 4 neighbours for `word[i + 1]`,
        then **unmark** it on the way back. The mistake people make: forgetting to unmark, so a later start finds
        cells blocked by an old attempt.

**⏱ Cost:** **O(rows · cols · 3ᴸ)** time for a word of length L (after the first step, at most 3 new directions) · **O(L)** space for the recursion.

In [ ]:
def exist(board, word):
    rows, cols = len(board), len(board[0])

    def dfs(r, c, i):
        if i == len(word):                     # matched every letter
            return True
        if not (0 <= r < rows and 0 <= c < cols) or board[r][c] != word[i]:
            return False                       # off the board, wrong letter, or already on the path ("#")
        board[r][c] = "#"                      # mark: this cell is on my path
        found = (dfs(r, c + 1, i + 1) or dfs(r + 1, c, i + 1) or
                 dfs(r, c - 1, i + 1) or dfs(r - 1, c, i + 1))
        board[r][c] = word[i]                  # unmark on the way back (undo)
        return found

    return any(dfs(r, c, 0) for r in range(rows) for c in range(cols))

B = [["A", "B", "C", "E"], ["S", "F", "C", "S"], ["A", "D", "E", "E"]]
assert exist(B, "ABCCED") is True
assert exist(B, "SEE") is True
assert exist(B, "ABCB") is False               # would need to reuse B
assert B == [["A", "B", "C", "E"], ["S", "F", "C", "S"], ["A", "D", "E", "E"]]   # board is left unchanged
assert exist([["a"]], "a") is True and exist([["a"]], "ab") is False
assert exist([["a", "a"]], "aaa") is False     # only two cells
print(exist(B, "ABCCED"))

In [ ]:
def word_search_movie(board, word):
    m = Movie("Word Search", f"Trace {word} through side-by-side squares, each square once")
    rows, cols = len(board), len(board[0])
    letters = [row[:] for row in board]        # a clean copy to draw (the real board gets # marks)
    path = []

    def pic(caption, extra=None):
        hi = {cell: "win" for cell in path}
        if path:
            hi[path[-1]] = "look"
        hi.update(extra or {})
        m.step(caption, grid(letters, "board (blue = my finger's path)", hi=hi),
               arr(list(word), "word", hi={k: "good" for k in range(len(path))},
                   ptr={"next": len(path)} if len(path) < len(word) else None))

    def dfs(r, c, i):
        if i == len(word):
            return True
        if not (0 <= r < rows and 0 <= c < cols):
            return False                       # off the board: nothing to show
        if board[r][c] == "#":
            pic(f"`{r},{c}` is already on my path — each square only once.", {(r, c): "bad"})
            return False
        if board[r][c] != word[i]:
            pic(f"`{letters[r][c]}` at {r},{c} is not the `{word[i]}` we need. Not this way.", {(r, c): "bad"})
            return False
        board[r][c] = "#"
        path.append((r, c))
        pic(f"`{word[i]}` matches at {r},{c}! **Mark** it. " +
            (f"Look around for `{word[i + 1]}`: right, down, left, up." if i + 1 < len(word) else "That was the last letter."))
        found = (dfs(r, c + 1, i + 1) or dfs(r + 1, c, i + 1) or
                 dfs(r, c - 1, i + 1) or dfs(r - 1, c, i + 1))
        board[r][c] = word[i]
        if not found:
            path.pop()
            pic(f"Stuck at {r},{c}. **Go back one step (undo)**: unmark it and try the next direction.",
                {(r, c): "bad"})
        return found

    pic(f"Find `{word}`. Start from each square in turn.")
    ok = any(dfs(r, c, 0) for r in range(rows) for c in range(cols))
    pic(f"Found **{word}**! ✅" if ok else f"No path spells {word}.", {cell: "good" for cell in path})
    return ok, m

board = [["A", "B", "C"], ["B", "C", "D"], ["C", "E", "F"]]
answer, m = word_search_movie([row[:] for row in board], "ABCE")
assert answer == exist([row[:] for row in board], "ABCE") is True
m.show()

---

## 70 · Number of Islands · [LeetCode 200](https://leetcode.com/problems/number-of-islands/) · 🟡 Medium

**🧸 Like you're five:** A map shows land and water. Walk over the map square by square. When you step on land nobody has
        coloured yet, you found a new island — pour paint on it and let the paint spread to all land it touches, so
        you never count that island again.

**The question:** Given an `m × n` grid of `"1"` (land) and `"0"` (water), return the number of islands. An island is land connected **up/down/left/right** (not diagonally); outside the grid is water.

| Input | Output |
|---|---|
| `grid = [["1","1","1","1","0"],["1","1","0","1","0"],["1","1","0","0","0"],["0","0","0","0","0"]]` | `1` |
| `grid = [["1","1","0","0","0"],["1","1","0","0","0"],["0","0","1","0","0"],["0","0","0","1","1"]]` | `3` |
| `grid = [["0"]]` | `0` |

**🔎 How to spot it:** "count the **connected groups** / regions / islands in a grid" → loop over every cell; each
        unvisited land cell starts a **flood fill** (BFS or DFS) that marks its whole group. Count the starts.

**💡 The trick (flood fill (bfs)):** Scan every cell. When you meet a `"1"`, add one to the count and flood from it, turning every
        connected `"1"` into `"0"` (or adding it to a `seen` set). Mark a cell **when you put it in the queue**, not
        when you take it out — otherwise the same cell gets queued many times. Use a queue (or an explicit stack)
        rather than recursion: a 300 × 300 all-land grid would overflow Python's recursion limit.

**⏱ Cost:** **O(rows · cols)** time (each cell is queued at most once) · **O(rows · cols)** space in the worst case for the queue. Union-find (see problem 78) is the other classic answer.

In [ ]:
import random
from collections import deque


def num_islands(grid):
    grid = [row[:] for row in grid]            # work on a copy, so the caller's map is untouched
    rows, cols = len(grid), len(grid[0])
    count = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] != "1":
                continue
            count += 1                         # land nobody has painted: a new island
            grid[r][c] = "0"
            q = deque([(r, c)])
            while q:                           # paint the whole island
                cr, cc = q.popleft()
                for nr, nc in ((cr + 1, cc), (cr - 1, cc), (cr, cc + 1), (cr, cc - 1)):
                    if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == "1":
                        grid[nr][nc] = "0"     # mark when you ADD, so it is never added twice
                        q.append((nr, nc))
    return count


def islands_slow(grid):                        # recursive DFS, only to check the answer on small maps
    seen = set()
    def sink(r, c):
        if 0 <= r < len(grid) and 0 <= c < len(grid[0]) and grid[r][c] == "1" and (r, c) not in seen:
            seen.add((r, c))
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                sink(r + dr, c + dc)
            return 1
        return 0
    return sum(sink(r, c) for r in range(len(grid)) for c in range(len(grid[0])))

G1 = [list("11110"), list("11010"), list("11000"), list("00000")]
G2 = [list("11000"), list("11000"), list("00100"), list("00011")]
assert num_islands(G1) == 1
assert num_islands(G2) == 3
assert num_islands([["0"]]) == 0 and num_islands([["1"]]) == 1
assert num_islands([list("101"), list("010"), list("101")]) == 5   # diagonals do not connect
rng = random.Random(0)
for _ in range(300):
    h, w = rng.randint(1, 6), rng.randint(1, 6)
    g = [[rng.choice("01") for _ in range(w)] for _ in range(h)]
    assert num_islands(g) == islands_slow(g)
print(num_islands(G2))

In [ ]:
def islands_movie(grid_in):
    m = Movie("Number of Islands", "Each new land square starts a paint flood; count the floods")
    g = [row[:] for row in grid_in]
    rows, cols = len(g), len(g[0])
    show = [row[:] for row in grid_in]         # what we draw: painted land shows its island's letter
    count, finished, cur = 0, set(), set()

    def pic(caption, ring=()):
        hi = {cell: "done" for cell in finished}
        hi.update({cell: "good" for cell in cur})
        hi.update({cell: "new" for cell in ring})
        m.step(caption, grid(show, "map (1 = land, 0 = water)", hi=hi, cmap={"1": "land", "0": "water"}),
               note(islands=count))

    pic("Walk the map row by row, left to right. Water and already-painted land are skipped.")
    for r in range(rows):
        for c in range(cols):
            if g[r][c] != "1":
                continue
            before = f"Island {'ABCDEFGH'[count - 1]} is finished — no more land touches it. " if count else ""
            finished |= cur
            count += 1
            name = "ABCDEFGH"[count - 1]
            g[r][c] = "0"; show[r][c] = name
            cur = {(r, c)}
            q = deque([(r, c)])
            pic(before + f"Unpainted land at {r},{c}: a **new island {name}**! Count = {count}. Pour paint here.", [(r, c)])
            while q:
                ring = []
                for _ in range(len(q)):
                    cr, cc = q.popleft()
                    for nr, nc in ((cr + 1, cc), (cr - 1, cc), (cr, cc + 1), (cr, cc - 1)):
                        if 0 <= nr < rows and 0 <= nc < cols and g[nr][nc] == "1":
                            g[nr][nc] = "0"; show[nr][nc] = name
                            q.append((nr, nc)); ring.append((nr, nc))
                if ring:
                    cur |= set(ring)
                    pic(f"The paint spreads one ring: {len(ring)} more land square(s) join island {name}.", ring)
    finished, cur = set(), finished | cur      # colour every island green for the last picture
    pic(f"The whole map is walked. **{count} islands** (A, B, C…). Each was painted by exactly one flood.")
    return count, m

G = [list("11000"), list("11000"), list("00100"), list("00011")]
answer, m = islands_movie(G)
assert answer == num_islands(G) == 3
m.show()

---

## 71 · Max Area of Island · [LeetCode 695](https://leetcode.com/problems/max-area-of-island/) · 🟡 Medium

**🧸 Like you're five:** Same map of land and water, but now you want the **biggest** island. Each time you find a new
        island, walk all over it and count your steps. Remember the biggest count.

**The question:** Given a binary grid (`1` = land, `0` = water), return the **area** (number of cells) of the largest island, where land connects up/down/left/right. Return `0` if there is no land.

| Input | Output |
|---|---|
| `the 8 × 13 grid from LeetCode` | `6` |
| `grid = [[0,0,0,0,0,0,0,0]]` | `0` |
| `grid = [[1,1],[1,0]]` | `3` |

**🔎 How to spot it:** "the **largest / size of** a connected region" → the island flood fill from problem 70, but the
        flood **returns how many cells** it painted. Keep the maximum.

**💡 The trick (flood fill + count):** Loop over the cells. From each unseen `1`, run a DFS with an explicit **stack**: pop a cell, add 1 to
        `area`, push its unseen land neighbours (marking them as you push). When the stack is empty, `best =
        max(best, area)`. Forgetting to reset `area` for each new island is the common bug.

**⏱ Cost:** **O(rows · cols)** time · **O(rows · cols)** space for `seen` and the stack. (You may mark by writing `0` into the grid instead of a `seen` set if changing the input is allowed.)

In [ ]:
import random


def max_area_of_island(grid):
    rows, cols = len(grid), len(grid[0])
    seen, best = set(), 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] != 1 or (r, c) in seen:
                continue
            seen.add((r, c))
            stack, area = [(r, c)], 0          # a new island: start counting from 0
            while stack:
                cr, cc = stack.pop()           # DFS: take the newest cell
                area += 1
                for nr, nc in ((cr + 1, cc), (cr - 1, cc), (cr, cc + 1), (cr, cc - 1)):
                    if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1 and (nr, nc) not in seen:
                        seen.add((nr, nc))
                        stack.append((nr, nc))
            best = max(best, area)
    return best


def area_slow(grid):                           # recursive version, only to check on small grids
    seen = set()
    def size(r, c):
        if not (0 <= r < len(grid) and 0 <= c < len(grid[0])) or grid[r][c] != 1 or (r, c) in seen:
            return 0
        seen.add((r, c))
        return 1 + size(r + 1, c) + size(r - 1, c) + size(r, c + 1) + size(r, c - 1)
    return max(size(r, c) for r in range(len(grid)) for c in range(len(grid[0])))

BIG = [[0,0,1,0,0,0,0,1,0,0,0,0,0],
       [0,0,0,0,0,0,0,1,1,1,0,0,0],
       [0,1,1,0,1,0,0,0,0,0,0,0,0],
       [0,1,0,0,1,1,0,0,1,0,1,0,0],
       [0,1,0,0,1,1,0,0,1,1,1,0,0],
       [0,0,0,0,0,0,0,0,0,0,1,0,0],
       [0,0,0,0,0,0,0,1,1,1,0,0,0],
       [0,0,0,0,0,0,0,1,1,0,0,0,0]]
assert max_area_of_island(BIG) == 6
assert max_area_of_island([[0, 0, 0, 0, 0, 0, 0, 0]]) == 0
assert max_area_of_island([[1, 1], [1, 0]]) == 3
assert max_area_of_island([[1]]) == 1
rng = random.Random(0)
for _ in range(300):
    h, w = rng.randint(1, 6), rng.randint(1, 6)
    g = [[rng.randint(0, 1) for _ in range(w)] for _ in range(h)]
    assert max_area_of_island(g) == area_slow(g)
print(max_area_of_island(BIG))

In [ ]:
def max_area_movie(grid_in):
    m = Movie("Max Area of Island", "Flood each island with a stack and count the squares")
    rows, cols = len(grid_in), len(grid_in[0])
    seen, best, finished, biggest = set(), 0, set(), set()

    def pic(caption, island=(), stack_=(), cur=None, area=0):
        hi = {cell: "done" for cell in finished}
        hi.update({cell: "win" for cell in island})
        hi.update({cell: "new" for cell in stack_})
        if cur:
            hi[cur] = "look"
        m.step(caption, grid(grid_in, "map (green = land, blue = counted)", hi=hi, cmap={1: "land"}),
               stack([f"{r},{c}" for r, c in stack_], "stack (to visit)"), note(area=area, best=best))

    pic("Find each island, walk all of it with a **stack**, and count its squares.")
    for r in range(rows):
        for c in range(cols):
            if grid_in[r][c] != 1 or (r, c) in seen:
                continue
            seen.add((r, c))
            stack_, area, island = [(r, c)], 0, set()
            pic(f"New island at {r},{c}. Put it on the stack; area starts at 0.", island, stack_)
            while stack_:
                cr, cc = stack_.pop()
                area += 1
                island.add((cr, cc))
                pushed = []
                for nr, nc in ((cr + 1, cc), (cr - 1, cc), (cr, cc + 1), (cr, cc - 1)):
                    if 0 <= nr < rows and 0 <= nc < cols and grid_in[nr][nc] == 1 and (nr, nc) not in seen:
                        seen.add((nr, nc)); stack_.append((nr, nc)); pushed.append(f"`{nr},{nc}`")
                pic(f"Pop {cr},{cc}: area = **{area}**. " +
                    (f"Push its new land neighbours {', '.join(pushed)}." if pushed else "No new land next to it."),
                    island, stack_, (cr, cc), area)
            if area > best:
                biggest = set(island)
            best = max(best, area)
            finished |= island
            pic(f"Stack empty: this island has area **{area}**. Best so far = {best}.", (), (), None, area)
    m.step(f"Every island counted. The biggest has area **{best}**.",
           grid(grid_in, "map", hi={cell: "good" for cell in biggest}, cmap={1: "land"}), note(best=best))
    return best, m

G = [[1, 1, 0, 0, 1],
     [1, 0, 0, 1, 1],
     [0, 0, 1, 1, 0],
     [1, 0, 0, 1, 0]]
answer, m = max_area_movie(G)
assert answer == max_area_of_island(G) == 6
m.show()

---

## 72 · Flood Fill · [LeetCode 733](https://leetcode.com/problems/flood-fill/) · 🟢 Easy

**🧸 Like you're five:** It's the paint-bucket tool in a drawing app. Click one square: it and every square of the **same
        colour** touching it (and touching those, and so on) change to the new colour. Squares of other colours stop
        the paint.

**The question:** Given an image `image[r][c]`, a start pixel `(sr, sc)` and a `color`, recolour the start pixel and every pixel connected to it **4-directionally** with the same original colour. Return the image.

| Input | Output |
|---|---|
| `image = [[1,1,1],[1,1,0],[1,0,1]], sr = 1, sc = 1, color = 2` | `[[2,2,2],[2,2,0],[2,0,1]]` |
| `image = [[0,0,0],[0,0,0]], sr = 0, sc = 0, color = 0` | `[[0,0,0],[0,0,0]]` |
| `image = [[5]], sr = 0, sc = 0, color = 7` | `[[7]]` |

**🔎 How to spot it:** "change a cell and **everything connected** to it that looks the same" → BFS/DFS from that one
        cell. (Problems 70 and 71 are this, run from many cells.)

**💡 The trick (bfs from one cell):** Remember the old colour. **If it already equals the new colour, return at once** — otherwise the
        fill keeps finding "old-coloured" cells it just painted and never ends; this is the famous trap. Else paint
        the start, put it in a queue, and for each cell taken out, paint and queue every neighbour that still has the
        old colour.

**⏱ Cost:** **O(rows · cols)** time · **O(rows · cols)** space for the queue in the worst case.

In [ ]:
from collections import deque


def flood_fill(image, sr, sc, color):
    old = image[sr][sc]
    if old == color:                           # already that colour: nothing to do (and no endless loop)
        return image
    rows, cols = len(image), len(image[0])
    image[sr][sc] = color
    q = deque([(sr, sc)])
    while q:
        r, c = q.popleft()
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < rows and 0 <= nc < cols and image[nr][nc] == old:
                image[nr][nc] = color          # paint when queued, so it is queued only once
                q.append((nr, nc))
    return image

assert flood_fill([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1, 2) == [[2, 2, 2], [2, 2, 0], [2, 0, 1]]
assert flood_fill([[0, 0, 0], [0, 0, 0]], 0, 0, 0) == [[0, 0, 0], [0, 0, 0]]
assert flood_fill([[5]], 0, 0, 7) == [[7]]
assert flood_fill([[0, 1], [1, 0]], 0, 0, 3) == [[3, 1], [1, 0]]   # a diagonal does not connect
print(flood_fill([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1, 2))

In [ ]:
def flood_fill_movie(image, sr, sc, color):
    m = Movie("Flood Fill", f"The paint bucket: pour colour {color} at {sr},{sc}")
    old = image[sr][sc]
    rows, cols = len(image), len(image[0])
    painted = set()

    def pic(caption, ring=(), final=False):
        hi = {cell: ("good" if final else "done") for cell in painted}
        hi.update({cell: "new" for cell in ring})
        m.step(caption, grid(image, "image", hi=hi), queue([f"{r},{c}" for r, c in q], "queue"),
               note(old=old, new=color))

    q = deque()
    pic(f"Pour colour `{color}` on {sr},{sc}. It holds `{old}`, so paint spreads only through touching `{old}`s.",
        [(sr, sc)])
    if old == color:
        return image, m
    image[sr][sc] = color
    painted.add((sr, sc))
    q.append((sr, sc))
    pic(f"Paint the start and put it in the queue.", [(sr, sc)])
    while q:
        ring = []
        for _ in range(len(q)):
            r, c = q.popleft()
            for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                if 0 <= nr < rows and 0 <= nc < cols and image[nr][nc] == old:
                    image[nr][nc] = color
                    q.append((nr, nc)); ring.append((nr, nc))
        painted |= set(ring)
        if ring:
            pic(f"The paint spreads one ring: {len(ring)} touching square(s) of colour `{old}` become `{color}`.", ring)
    cut_off = [(r, c) for r in range(rows) for c in range(cols) if image[r][c] == old]
    m.step(f"Nothing left to spread to. Done! The red `{old}`s have the old colour too, but other colours cut "
           f"them off, so the paint never reached them.",
           grid(image, "image", hi={**{cell: "good" for cell in painted}, **{cell: "bad" for cell in cut_off}}),
           note(old=old, new=color))
    return image, m

img = [[1, 1, 1, 0, 1],
       [2, 0, 1, 0, 1],
       [1, 1, 1, 2, 2],
       [1, 0, 0, 1, 1]]
answer, m = flood_fill_movie([row[:] for row in img], 2, 2, 3)
assert answer == flood_fill([row[:] for row in img], 2, 2, 3) == [[3, 3, 3, 0, 1], [2, 0, 3, 0, 1],
                                                                  [3, 3, 3, 2, 2], [3, 0, 0, 1, 1]]
m.show()

---

## 73 · Rotting Oranges · [LeetCode 994](https://leetcode.com/problems/rotting-oranges/) · 🟡 Medium

**🧸 Like you're five:** A box of oranges. Every minute, each rotten orange makes the fresh oranges right next to it (up,
        down, left, right) rotten too. All the rotten ones spread **at the same time**, like several ripples at once.
        How many minutes until no fresh orange is left?

**The question:** In a grid, `0` = empty, `1` = fresh orange, `2` = rotten orange. Each minute, fresh oranges 4-directionally next to a rotten one become rotten. Return the minimum minutes until no fresh orange remains, or `-1` if that is impossible.

| Input | Output |
|---|---|
| `grid = [[2,1,1],[1,1,0],[0,1,1]]` | `4` |
| `grid = [[2,1,1],[0,1,1],[1,0,1]]` | `-1` |
| `grid = [[0,2]]` | `0` |

**🔎 How to spot it:** "spreads **every minute** to the neighbours" / "**minimum time** for everything to be reached"
        with **several starting points** → **multi-source BFS**: put *all* the sources in the queue before you
        start, then go ring by ring.

**💡 The trick (multi-source bfs):** Queue every rotten orange at the start and count the fresh ones. Each loop round is one minute: take
        exactly the oranges that are in the queue **now** (`for _ in range(len(q))`), rot their fresh neighbours, and
        queue those. Stop when the queue is empty or nothing fresh is left. If fresh oranges remain, return `-1`.
        Two traps: running a separate BFS from each rotten orange (wrong, they spread together), and counting one
        extra minute at the end (stop as soon as `fresh == 0`).

**⏱ Cost:** **O(rows · cols)** time — each orange enters the queue once · **O(rows · cols)** space. Simulating the whole grid minute by minute is O((rows · cols)²).

In [ ]:
import random
from collections import deque


def oranges_rotting(grid):
    grid = [row[:] for row in grid]
    rows, cols = len(grid), len(grid[0])
    q = deque((r, c) for r in range(rows) for c in range(cols) if grid[r][c] == 2)   # ALL rotten start together
    fresh = sum(row.count(1) for row in grid)
    minutes = 0
    while q and fresh:
        minutes += 1
        for _ in range(len(q)):                # exactly the oranges that are rotten right now
            r, c = q.popleft()
            for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                    grid[nr][nc] = 2
                    fresh -= 1
                    q.append((nr, nc))
    return minutes if fresh == 0 else -1


def rotting_slow(grid):                        # minute-by-minute simulation of the whole box
    g, minutes = [row[:] for row in grid], 0
    while True:
        rot = [(r, c) for r in range(len(g)) for c in range(len(g[0])) if g[r][c] == 1 and any(
            0 <= r + dr < len(g) and 0 <= c + dc < len(g[0]) and g[r + dr][c + dc] == 2
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)))]
        if not rot:
            return -1 if any(1 in row for row in g) else minutes
        for r, c in rot:
            g[r][c] = 2
        minutes += 1

assert oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]) == 4
assert oranges_rotting([[2, 1, 1], [0, 1, 1], [1, 0, 1]]) == -1   # bottom-left orange is cut off
assert oranges_rotting([[0, 2]]) == 0                            # nothing fresh: zero minutes
assert oranges_rotting([[0]]) == 0 and oranges_rotting([[1]]) == -1
rng = random.Random(0)
for _ in range(300):
    h, w = rng.randint(1, 5), rng.randint(1, 5)
    g = [[rng.choice([0, 1, 1, 2]) for _ in range(w)] for _ in range(h)]
    assert oranges_rotting(g) == rotting_slow(g)
print(oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]))

In [ ]:
def rotting_movie(grid_in):
    m = Movie("Rotting Oranges", "Every rotten orange spreads at the same time: one ring per minute")
    g = [row[:] for row in grid_in]
    rows, cols = len(g), len(g[0])
    face = {0: None, 1: "🍊", 2: "🤢"}

    def pic(caption, new=(), final=False):
        hi = {(r, c): "bad" for r in range(rows) for c in range(cols) if g[r][c] == 2}
        hi.update({cell: "new" for cell in new})
        if final:                              # green: the answer (last to go bad, or never reachable)
            hi.update({(r, c): "good" for r in range(rows) for c in range(cols) if g[r][c] == 1})
            hi.update({cell: "good" for cell in new})
        m.step(caption, grid([[face[x] for x in row] for row in g], "the box", hi=hi),
               queue([f"{r},{c}" for r, c in q], "queue (rotten, will spread)"),
               note(minute=minutes, fresh=fresh))

    q = deque((r, c) for r in range(rows) for c in range(cols) if g[r][c] == 2)
    fresh = sum(row.count(1) for row in g)
    minutes, new = 0, []
    pic(f"Minute 0: {len(q)} rotten oranges, {fresh} fresh. **All** rotten ones go into the queue together.")
    while q and fresh:
        minutes += 1
        new = []
        for _ in range(len(q)):
            r, c = q.popleft()
            for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                if 0 <= nr < rows and 0 <= nc < cols and g[nr][nc] == 1:
                    g[nr][nc] = 2; fresh -= 1
                    q.append((nr, nc)); new.append((nr, nc))
        pic(f"Minute {minutes}: every orange that was rotten spreads one step. {len(new)} more go bad; "
            f"{fresh} fresh left.", new)
    new = new if fresh == 0 else []
    answer = minutes if fresh == 0 else -1
    pic(f"No fresh orange is left after **{answer}** minutes. Green = the last orange to go bad." if answer >= 0 else
        f"The queue is empty but {fresh} fresh orange(s) can never be reached (green): answer **-1**.",
        new, final=True)
    return answer, m

box = [[2, 1, 1, 0, 1],
       [0, 1, 0, 1, 1],
       [1, 1, 1, 1, 2]]
answer, m = rotting_movie(box)
assert answer == oranges_rotting(box) == 4
m.show()

---

## 74 · Clone Graph · [LeetCode 133](https://leetcode.com/problems/clone-graph/) · 🟡 Medium

**🧸 Like you're five:** You have a group of friends, each holding a list of who their friends are. Build an exact twin
        group: a twin for every person, and each twin's friends must be **twins**, never the original people. Keep a
        notebook "person → their twin" so you never make the same twin twice.

**The question:** Given a reference to one node of a connected undirected graph (each `Node` has `val` and a list `neighbors`), return a **deep copy**: new nodes with the same values and the same connections, sharing no node with the original. Values are `1..n`; an empty graph is `None`.

| Input | Output |
|---|---|
| `adjList = [[2,4],[1,3],[2,4],[1,3]]` | `[[2,4],[1,3],[2,4],[1,3]]` |
| `adjList = [[]]` | `[[]]` |
| `adjList = []` | `[]` |

**🔎 How to spot it:** "**copy / clone** a structure that has **cycles or shared nodes**" (graph, random-pointer list) →
        a dictionary **old node → new node**, filled while you walk (BFS or DFS). The dictionary is also your
        "visited" set.

**💡 The trick (bfs + old→new dictionary):** Make the twin of the start node and queue it. Pop a node; for each neighbour: if it has no twin
        yet, make one and queue the neighbour; then append **the neighbour's twin** to the current twin's list. The
        classic bug: appending the *original* neighbour, so the copy secretly points back into the old graph.

**⏱ Cost:** **O(V + E)** time — every node and edge once · **O(V)** space for the dictionary and the queue.

In [ ]:
from collections import deque


class Node:
    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []


def clone_graph(node):
    if node is None:
        return None
    twin = {node: Node(node.val)}              # old node -> its new twin (also our "visited")
    q = deque([node])
    while q:
        cur = q.popleft()
        for nb in cur.neighbors:
            if nb not in twin:                 # first meeting: make its twin, visit it later
                twin[nb] = Node(nb.val)
                q.append(nb)
            twin[cur].neighbors.append(twin[nb])   # wire twin to twin — never to an old node
    return twin[node]


def build_graph(adj):                          # LeetCode's adjacency list (1-indexed) -> node 1
    nodes = [Node(i + 1) for i in range(len(adj))]
    for i, friends in enumerate(adj):
        nodes[i].neighbors = [nodes[j - 1] for j in friends]
    return nodes[0] if nodes else None


def all_nodes(node):                           # every node reachable from `node`
    seen, q = {node}, deque([node])
    while q:
        for nb in q.popleft().neighbors:
            if nb not in seen:
                seen.add(nb); q.append(nb)
    return seen


def to_adj(node):                              # node -> adjacency list, LeetCode's format
    if node is None:
        return []
    return [[nb.val for nb in n.neighbors] for n in sorted(all_nodes(node), key=lambda n: n.val)]

for adj in ([[2, 4], [1, 3], [2, 4], [1, 3]], [[]], [[2], [1]], [[2, 3], [1, 3], [1, 2]]):
    old = build_graph(adj)
    new = clone_graph(old)
    assert to_adj(new) == adj                                   # same shape
    assert not {id(n) for n in all_nodes(old)} & {id(n) for n in all_nodes(new)}   # no shared node
assert clone_graph(None) is None
print(to_adj(clone_graph(build_graph([[2, 4], [1, 3], [2, 4], [1, 3]]))))

In [ ]:
def clone_movie(start, pos):
    m = Movie("Clone Graph", "Make a twin for every node; twins only hold hands with twins")
    olds = sorted(all_nodes(start), key=lambda n: n.val)
    old_edges = [(a.val, b.val) for a in olds for b in a.neighbors if a.val < b.val]
    twin = {start: Node(start.val)}
    q = deque([start])
    done = set()

    def pic(caption, cur=None, nb=None, wire=None, final=False):
        made = sorted(t.val for t in twin.values())
        wired = sorted({tuple(sorted((a.val, b.val))) for t in twin.values() for a, b in
                        ((t, x) for x in t.neighbors)})
        hi_old = {v: "done" for v in done}
        if cur: hi_old[cur.val] = "look"
        if nb: hi_old[nb.val] = "new"
        m.step(caption,
               graph([n.val for n in olds], old_edges, "the old graph", hi=hi_old,
                     ehi={(cur.val, nb.val): "look"} if cur and nb else None, pos=pos),
               graph([f"{v}'" for v in made], [(f"{a}'", f"{b}'") for a, b in wired], "the copy (new nodes)",
                     hi={f"{v}'": "good" for v in made} if final else {f"{nb.val}'": "new"} if nb else None,
                     ehi={(f"{wire[0]}'", f"{wire[1]}'"): "new"} if wire else None,
                     pos={f"{v}'": pos[v] for v in made}),
               hmap({o.val: f"{t.val}'" for o, t in twin.items()}, "twin notebook (old → new)"),
               queue([n.val for n in q], "queue"))

    pic(f"Make the twin `{start.val}'` of the start node and write `{start.val} → {start.val}'` in the notebook.")
    while q:
        cur = q.popleft()
        for nb in cur.neighbors:
            if nb not in twin:
                twin[nb] = Node(nb.val)
                q.append(nb)
                first = f"`{nb.val}` has no twin yet: make `{nb.val}'` and queue `{nb.val}`. "
            else:
                first = f"`{nb.val}` already has twin `{nb.val}'` (the notebook says so). "
            twin[cur].neighbors.append(twin[nb])
            pic(f"At `{cur.val}`, friend `{nb.val}`. " + first + f"Wire `{cur.val}'` → `{nb.val}'`.",
                cur, nb, (cur.val, nb.val))
        done.add(cur.val)
    pic("Queue empty. The copy has the same shape, and every node in it is **new** — a deep copy. ✅",
        final=True)
    return twin[start], m

g = build_graph([[2, 4], [1, 3], [2, 4], [1, 3]])
pos = {1: (0.1, 0.1), 2: (0.9, 0.1), 3: (0.9, 0.9), 4: (0.1, 0.9)}
copy, m = clone_movie(g, pos)
assert to_adj(copy) == to_adj(clone_graph(g)) == [[2, 4], [1, 3], [2, 4], [1, 3]]
assert not {id(n) for n in all_nodes(g)} & {id(n) for n in all_nodes(copy)}
m.show()

---

## 75 · Course Schedule · [LeetCode 207](https://leetcode.com/problems/course-schedule/) · 🟡 Medium

**🧸 Like you're five:** To play football you must first learn to kick, and to kick you must first learn to stand. Start
        with the things that need nothing first. Each time you learn one, cross it off the "still needed" lists of
        the others. If some things keep waiting on each other in a circle, you can never learn them.

**The question:** There are `numCourses` courses `0..n-1`. `prerequisites[i] = [a, b]` means you must take `b` before `a`. Return `True` if you can finish every course (that is, there is **no cycle**).

| Input | Output |
|---|---|
| `numCourses = 2, prerequisites = [[1,0]]` | `True` |
| `numCourses = 2, prerequisites = [[1,0],[0,1]]` | `False` |
| `numCourses = 1, prerequisites = []` | `True` |

**🔎 How to spot it:** "tasks with **prerequisites / dependencies / must come before**" → a directed graph.
        "Can you finish all?" = "is there **no cycle**?" → topological sort. (Recipes, build systems, spreadsheet
        cells.)

**💡 The trick (topological sort (kahn)):** Draw an arrow `b → a` for each pair and count `need[a]`, how many arrows point into `a`. Queue every
        course with `need == 0`. Pop one, "take" it, and for each arrow out of it do `need[next] -= 1`; when that
        reaches 0, queue it. Courses on a cycle never reach 0, so at the end `taken == numCourses` means no cycle.
        Watch the direction: `[a, b]` means **b first**, the arrow goes `b → a`.

**⏱ Cost:** **O(V + E)** time · **O(V + E)** space for the lists. (DFS with three colours — unvisited / on the path / finished — is the other classic way.)

In [ ]:
import random
from collections import deque


def can_finish(num_courses, prerequisites):
    after = [[] for _ in range(num_courses)]   # after[b] = courses that need b first
    need = [0] * num_courses                   # need[a] = how many courses a still waits for
    for a, b in prerequisites:
        after[b].append(a)
        need[a] += 1
    q = deque(c for c in range(num_courses) if need[c] == 0)   # free to take right now
    taken = 0
    while q:
        c = q.popleft()
        taken += 1
        for nxt in after[c]:
            need[nxt] -= 1                     # one less thing to wait for
            if need[nxt] == 0:
                q.append(nxt)
    return taken == num_courses                # anything left over sits on a cycle


def has_cycle(n, prerequisites):               # DFS with colours, only to check the answer
    after = [[] for _ in range(n)]
    for a, b in prerequisites:
        after[b].append(a)
    colour = [0] * n                           # 0 = new, 1 = on my path, 2 = finished
    def dfs(u):
        colour[u] = 1
        for v in after[u]:
            if colour[v] == 1 or (colour[v] == 0 and dfs(v)):
                return True
        colour[u] = 2
        return False
    return any(colour[u] == 0 and dfs(u) for u in range(n))

assert can_finish(2, [[1, 0]]) is True
assert can_finish(2, [[1, 0], [0, 1]]) is False
assert can_finish(1, []) is True
assert can_finish(1, [[0, 0]]) is False        # a course that needs itself
assert can_finish(3, [[1, 0], [1, 0], [2, 1]]) is True   # duplicate pair is fine
rng = random.Random(0)
for _ in range(300):
    n = rng.randint(1, 6)
    pre = [[rng.randrange(n), rng.randrange(n)] for _ in range(rng.randint(0, 7))]
    assert can_finish(n, pre) == (not has_cycle(n, pre))
print(can_finish(2, [[1, 0]]))

In [ ]:
def course_movie(m, n, prerequisites, pos, want_order=False):   # m = the Movie to record into
    after = [[] for _ in range(n)]
    need = [0] * n
    for a, b in prerequisites:
        after[b].append(a)
        need[a] += 1
    arrows = [(b, a) for a, b in prerequisites]
    q = deque(c for c in range(n) if need[c] == 0)
    order, crossed = [], {}

    def pic(caption, cur=None, stuck=False, final=False):
        hi = {c: "done" for c in order}
        hi.update({c: "new" for c in q})
        if cur is not None:
            hi[cur] = "look"
        if stuck:
            hi.update({c: "bad" for c in range(n) if c not in order})
        if final:
            hi.update({c: "good" for c in order})
        m.step(caption, graph(range(n), arrows, "b → a  means  take b first", hi=hi, ehi=crossed,
                              directed=True, pos=pos),
               hmap({c: need[c] for c in range(n)}, "need (still waiting for)", hi={c: "look" for c in q} or None),
               queue(list(q), "ready queue"), arr(order, "taken, in order", index=False))

    pic(f"Count the arrows into each course. {', '.join(map(str, q))} need nothing: they go in the queue.")
    while q:
        c = q.popleft()
        order.append(c)
        pic(f"Take course **{c}** from the front of the queue — it waits for nothing." +
            ("" if after[c] else " No arrows leave it, so nobody else changes."), c)
        if not after[c]:
            continue
        freed = []
        for nxt in after[c]:
            need[nxt] -= 1
            crossed[(c, nxt)] = "done"
            if need[nxt] == 0:
                q.append(nxt)
                freed.append(nxt)
        tail = (f"Now {', '.join(map(str, freed))} wait for nothing → into the queue." if freed else
                "They still wait for something else.")
        pic(f"Cross off {c}'s arrows: " + ", ".join(f"`need[{x}]` -1" for x in after[c]) + f". {tail}", c)
    if len(order) < n:
        stuck = [c for c in range(n) if c not in order]
        pic(f"The queue is empty but {', '.join(map(str, stuck))} still wait — on each other, in a **circle**. "
            f"Can't finish: **False**.", stuck=True)
        return ([] if want_order else False), m
    pic(f"All {n} courses taken" + (f" in the order **{order}**." if want_order else ": no circle, so **True**."),
        final=True)
    return (order if want_order else True), m

pre = [[1, 0], [5, 0], [2, 1], [3, 2], [4, 3], [2, 4]]
pos = {0: (0, 0.5), 1: (0.3, 0.1), 5: (0.3, 0.9), 2: (0.6, 0.1), 3: (1, 0.1), 4: (0.8, 0.9)}
m = Movie("Course Schedule", "Take what needs nothing; cross it off; repeat")
answer, m = course_movie(m, 6, pre, pos)
assert answer == can_finish(6, pre) is False
m.show()

---

## 76 · Course Schedule II · [LeetCode 210](https://leetcode.com/problems/course-schedule-ii/) · 🟡 Medium

**🧸 Like you're five:** Same courses and the same "learn this first" rules — but now write down **the order** in which you
        take them. If a circle makes it impossible, hand back an empty list.

**The question:** Same input as problem 75. Return **an order** in which you can take all courses. If several orders work, return any of them; if it is impossible, return `[]`.

| Input | Output |
|---|---|
| `numCourses = 2, prerequisites = [[1,0]]` | `[0,1]` |
| `numCourses = 4, prerequisites = [[1,0],[2,0],[3,1],[3,2]]` | `[0,1,2,3] or [0,2,1,3]` |
| `numCourses = 1, prerequisites = []` | `[0]` |

**🔎 How to spot it:** "return **an order** that respects the dependencies" → topological sort; the order you pop from
        the queue *is* the answer. "Any valid order" in the question = test with a checker, not one fixed list.

**💡 The trick (topological sort (kahn)):** Exactly problem 75, but append each popped course to `order`. At the end return `order` if it has
        all `n` courses, else `[]` (a cycle). Many orders can be right: Kahn's algorithm with a plain queue gives one,
        a heap instead of a queue would give the smallest-numbers-first order.

**⏱ Cost:** **O(V + E)** time · **O(V + E)** space.

In [ ]:
import random
from collections import deque


def find_order(num_courses, prerequisites):
    after = [[] for _ in range(num_courses)]
    need = [0] * num_courses
    for a, b in prerequisites:
        after[b].append(a)
        need[a] += 1
    q = deque(c for c in range(num_courses) if need[c] == 0)
    order = []
    while q:
        c = q.popleft()
        order.append(c)                        # the pop order is a valid course order
        for nxt in after[c]:
            need[nxt] -= 1
            if need[nxt] == 0:
                q.append(nxt)
    return order if len(order) == num_courses else []   # short = a cycle blocked some courses


def is_valid_order(n, prerequisites, order):   # checks ANY correct answer, not one fixed list
    if sorted(order) != list(range(n)):
        return False
    where = {c: i for i, c in enumerate(order)}
    return all(where[b] < where[a] for a, b in prerequisites)

assert find_order(2, [[1, 0]]) == [0, 1]
assert is_valid_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]], find_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]]))
assert find_order(1, []) == [0]
assert find_order(2, [[1, 0], [0, 1]]) == []   # a circle: impossible
rng = random.Random(1)
for _ in range(300):
    n = rng.randint(1, 6)
    pre = [[rng.randrange(n), rng.randrange(n)] for _ in range(rng.randint(0, 7))]
    order = find_order(n, pre)
    assert (order == []) == has_cycle(n, pre)
    assert order == [] or is_valid_order(n, pre, order)
print(find_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]]))

In [ ]:
# the same movie function as problem 75 (course_movie), asked for the order this time
pre = [[1, 0], [2, 0], [3, 1], [3, 2], [4, 3], [5, 2]]
pos = {0: (0, 0.5), 1: (0.33, 0.05), 2: (0.33, 0.95), 3: (0.66, 0.5), 5: (0.72, 1), 4: (1, 0.2)}
m = Movie("Course Schedule II", "Take what needs nothing; the order you take them is the answer")
answer, m = course_movie(m, 6, pre, pos, want_order=True)
assert answer == find_order(6, pre) and is_valid_order(6, pre, answer)
m.show()

---

## 77 · Pacific Atlantic Water Flow · [LeetCode 417](https://leetcode.com/problems/pacific-atlantic-water-flow/) · 🟡 Medium

**🧸 Like you're five:** An island of hills. Rain runs downhill (or across flat ground). The Pacific Ocean touches the top
        and left edges, the Atlantic the bottom and right edges. Which squares send rain to **both** oceans? Instead
        of following every raindrop down, let each ocean **climb up** the hills and see how far it gets.

**The question:** Given an `m × n` grid of `heights`, water flows from a cell to a neighbour (4-directional) with height **less than or equal**. The Pacific touches the top and left edges; the Atlantic the bottom and right edges. Return every `[r, c]` from which water can reach **both** oceans (any order).

| Input | Output |
|---|---|
| `heights = [[1,2,2,3,5],[3,2,3,4,4],[2,4,5,3,1],[6,7,1,4,5],[5,1,1,2,4]]` | `[[0,4],[1,3],[1,4],[2,2],[3,0],[3,1],[4,0]]` |
| `heights = [[1]]` | `[[0,0]]` |
| `heights = [[1,1],[1,1]]` | `all four cells` |

**🔎 How to spot it:** "which cells can **reach** the border / the exit / both sides" → turn it around: start from the
        **targets** and search **backwards** (uphill). One search per target, then intersect the two sets.

**💡 The trick (bfs backwards from both oceans):** Run a BFS from all Pacific-edge cells at once, stepping to a neighbour only if it is **at least as
        high** (water could flow from there down to us). Do the same from the Atlantic edges. The answer is the cells
        in **both** sets. Forward search from every cell also works but repeats the same work: O((m·n)²).

**⏱ Cost:** **O(m · n)** time (each cell enters each BFS once) · **O(m · n)** space. Brute force (a search from every cell) is O((m · n)²).

In [ ]:
import random
from collections import deque


def pacific_atlantic(heights):
    rows, cols = len(heights), len(heights[0])

    def climb(starts):                         # every cell the ocean can reach by going UPHILL
        seen, q = set(starts), deque(starts)
        while q:
            r, c = q.popleft()
            for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                if (0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in seen
                        and heights[nr][nc] >= heights[r][c]):   # water could flow from there to here
                    seen.add((nr, nc))
                    q.append((nr, nc))
        return seen

    pacific = climb([(r, 0) for r in range(rows)] + [(0, c) for c in range(1, cols)])
    atlantic = climb([(r, cols - 1) for r in range(rows)] + [(rows - 1, c) for c in range(cols - 1)])
    return [[r, c] for r, c in sorted(pacific & atlantic)]


def pacific_atlantic_slow(heights):            # follow the water DOWN from every cell
    rows, cols = len(heights), len(heights[0])
    out = []
    for sr in range(rows):
        for sc in range(cols):
            seen, stack, pac, atl = {(sr, sc)}, [(sr, sc)], False, False
            while stack:
                r, c = stack.pop()
                pac |= r == 0 or c == 0
                atl |= r == rows - 1 or c == cols - 1
                for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                    if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in seen and heights[nr][nc] <= heights[r][c]:
                        seen.add((nr, nc)); stack.append((nr, nc))
            if pac and atl:
                out.append([sr, sc])
    return out

H = [[1, 2, 2, 3, 5], [3, 2, 3, 4, 4], [2, 4, 5, 3, 1], [6, 7, 1, 4, 5], [5, 1, 1, 2, 4]]
assert sorted(pacific_atlantic(H)) == [[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]
assert pacific_atlantic([[1]]) == [[0, 0]]
assert sorted(pacific_atlantic([[1, 1], [1, 1]])) == [[0, 0], [0, 1], [1, 0], [1, 1]]
rng = random.Random(0)
for _ in range(200):
    h, w = rng.randint(1, 5), rng.randint(1, 5)
    g = [[rng.randint(0, 5) for _ in range(w)] for _ in range(h)]
    assert sorted(pacific_atlantic(g)) == sorted(pacific_atlantic_slow(g))
print(pacific_atlantic(H))

In [ ]:
def pacific_atlantic_movie(heights):
    m = Movie("Pacific Atlantic Water Flow", "Each ocean climbs uphill; keep the squares both oceans reach")
    rows, cols = len(heights), len(heights[0])
    reach = {"Pacific": set(), "Atlantic": set()}

    def pic(caption, ocean=None, ring=()):
        panels = []
        for name, side in (("Pacific", "top & left"), ("Atlantic", "bottom & right")):
            hi = {cell: "win" for cell in reach[name]}
            if name == ocean:
                hi.update({cell: "new" for cell in ring})
            panels.append(grid(heights, f"{name} ({side}) reaches", hi=hi))
        m.step(caption, *panels)

    starts = {"Pacific": [(r, 0) for r in range(rows)] + [(0, c) for c in range(1, cols)],
              "Atlantic": [(r, cols - 1) for r in range(rows)] + [(rows - 1, c) for c in range(cols - 1)]}
    pic("Numbers are heights. Water flows to a neighbour that is lower or equal. Let each ocean climb **up**.")
    for ocean in ("Pacific", "Atlantic"):
        seen, q = set(starts[ocean]), deque(starts[ocean])
        reach[ocean] = set(seen)
        pic(f"The {ocean} touches its edge squares: rain there drops straight in.", ocean, starts[ocean])
        while q:
            ring = []
            for _ in range(len(q)):
                r, c = q.popleft()
                for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                    if (0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in seen
                            and heights[nr][nc] >= heights[r][c]):
                        seen.add((nr, nc)); q.append((nr, nc)); ring.append((nr, nc))
            reach[ocean] |= set(ring)
            if ring:
                pic(f"The {ocean} climbs one ring: {len(ring)} square(s) at least as high as a reached neighbour.",
                    ocean, ring)
        pic(f"The {ocean} can't climb any higher. Blue = rain from here reaches the {ocean}.")
    both = sorted(reach["Pacific"] & reach["Atlantic"])
    m.step(f"Keep the squares that are blue in **both** maps: {len(both)} squares send rain to both oceans.",
           grid(heights, "reaches both oceans", hi={cell: "good" for cell in both}))
    return [[r, c] for r, c in both], m

heights = [[1, 2, 2, 3], [3, 2, 3, 4], [2, 4, 5, 3], [6, 7, 1, 4]]
answer, m = pacific_atlantic_movie(heights)
assert sorted(answer) == sorted(pacific_atlantic(heights))
m.show()

---

## 78 · Number of Provinces · [LeetCode 547](https://leetcode.com/problems/number-of-provinces/) · 🟡 Medium

**🧸 Like you're five:** Every kid starts as the boss of their own team. When two kids are friends, their two teams join and
        one boss becomes the boss of both. At the end, count how many bosses are left — that is how many friend
        groups there are.

**The question:** `isConnected` is an `n × n` matrix; `isConnected[i][j] = 1` means cities `i` and `j` are directly connected. A **province** is a group of cities connected directly or indirectly. Return the number of provinces.

| Input | Output |
|---|---|
| `isConnected = [[1,1,0],[1,1,0],[0,0,1]]` | `2` |
| `isConnected = [[1,0,0],[0,1,0],[0,0,1]]` | `3` |
| `isConnected = [[1]]` | `1` |

**🔎 How to spot it:** "how many **groups / connected components**", "are these two in the same group", links arriving
        one at a time → **union-find** (disjoint set). A DFS/BFS from each unvisited city works too.

**💡 The trick (union-find (or dfs)):** `parent[i] = i` at first: everyone is their own boss. `find(x)` follows parents up to the boss
        (and shortcuts the path on the way — *path compression*). For every `1` in the matrix, `union(i, j)`: if the
        bosses differ, make one the parent of the other and do `groups -= 1`. The matrix is symmetric, so only look
        at `j > i`. Common slip: comparing `parent[i] == parent[j]` instead of `find(i) == find(j)`.

**⏱ Cost:** **O(n² · α(n))** ≈ O(n²) time — reading the matrix dominates · **O(n)** space. DFS is also O(n²) here.

In [ ]:
import random


def find_circle_num(is_connected):
    n = len(is_connected)
    parent = list(range(n))                    # everyone starts as their own boss

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]      # path compression: jump to the grandparent
            x = parent[x]
        return x

    groups = n
    for i in range(n):
        for j in range(i + 1, n):              # the matrix is symmetric: the upper half is enough
            if is_connected[i][j]:
                bi, bj = find(i), find(j)
                if bi != bj:                   # two different teams join
                    parent[bj] = bi
                    groups -= 1
    return groups


def provinces_dfs(is_connected):               # the other classic answer, used as a check
    n, seen, count = len(is_connected), set(), 0
    for s in range(n):
        if s in seen:
            continue
        count += 1
        stack = [s]; seen.add(s)
        while stack:
            u = stack.pop()
            for v in range(n):
                if is_connected[u][v] and v not in seen:
                    seen.add(v); stack.append(v)
    return count

assert find_circle_num([[1, 1, 0], [1, 1, 0], [0, 0, 1]]) == 2
assert find_circle_num([[1, 0, 0], [0, 1, 0], [0, 0, 1]]) == 3
assert find_circle_num([[1]]) == 1
rng = random.Random(0)
for _ in range(300):
    n = rng.randint(1, 8)
    M = [[1 if i == j else 0 for j in range(n)] for i in range(n)]
    for _ in range(rng.randint(0, n)):
        i, j = rng.randrange(n), rng.randrange(n)
        M[i][j] = M[j][i] = 1
    assert find_circle_num(M) == provinces_dfs(M)
print(find_circle_num([[1, 1, 0], [1, 1, 0], [0, 0, 1]]))

In [ ]:
def provinces_movie(M, pos):
    m = Movie("Number of Provinces", "Union-find: every city starts as its own boss")
    n = len(M)
    parent = list(range(n))
    links = [(i, j) for i in range(n) for j in range(i + 1, n) if M[i][j]]
    joined = {}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def pic(caption, i=None, j=None, final=False):
        hi = {c: "good" for c in range(n)} if final else {c: "look" for c in (i, j) if c is not None}
        m.step(caption, grid(M, "isConnected", hi={(i, j): "look"} if i is not None else None,
                             row_labels=list(range(n)), col_labels=list(range(n))),
               graph(range(n), links, "cities", hi=hi, ehi=joined, pos=pos),
               hmap({c: parent[c] for c in range(n)}, "parent (city → who it answers to)",
                    hi={c: "look" for c in (i, j) if c is not None}),
               note(provinces=groups))

    groups = n
    pic(f"{n} cities, and every city is its own boss: {n} provinces. Read the 1s above the diagonal.")
    for i, j in links:
        bi, bj = find(i), find(j)
        if bi != bj:
            parent[bj] = bi
            groups -= 1
            joined[(i, j)] = "good"
            pic(f"Cities {i} and {j} are linked. Their bosses {bi} and {bj} differ → join: {bi} becomes "
                f"{bj}'s boss. **Provinces = {groups}**.", i, j)
        else:
            joined[(i, j)] = "done"
            pic(f"Cities {i} and {j} are linked, but both already answer to boss {bi}. Same province — "
                f"nothing changes.", i, j)
    bosses = sorted({find(c) for c in range(n)})
    pic(f"Done. The bosses left are {bosses}: **{groups} provinces**.", final=True)
    return groups, m

M = [[1, 1, 1, 0, 0, 0],
     [1, 1, 1, 0, 0, 0],
     [1, 1, 1, 0, 0, 0],
     [0, 0, 0, 1, 1, 0],
     [0, 0, 0, 1, 1, 0],
     [0, 0, 0, 0, 0, 1]]
pos = {0: (0.1, 0.1), 1: (0.45, 0.1), 2: (0.27, 0.9), 3: (0.7, 0.1), 4: (0.95, 0.5), 5: (0.7, 0.9)}
answer, m = provinces_movie(M, pos)
assert answer == find_circle_num(M) == 3
m.show()

---

## 79 · Implement Trie (Prefix Tree) · [LeetCode 208](https://leetcode.com/problems/implement-trie-prefix-tree/) · 🟡 Medium

**🧸 Like you're five:** A word tree. From the root you walk down one letter at a time: c, then a, then t. Words that start
        the same share the same branch, like "cat" and "car" sharing "ca". A little tick ✓ on a node means "a real word
        ends here".

**The question:** Implement `Trie` with `insert(word)`, `search(word)` → `True` only if that exact word was inserted, and `startsWith(prefix)` → `True` if any inserted word starts with `prefix`. Lowercase letters only.

| Input | Output |
|---|---|
| `insert("apple"); search("apple"), search("app")` | `True, False` |
| `startsWith("app"); insert("app"); search("app")` | `True, True` |
| `search("") on an empty trie` | `False` |

**🔎 How to spot it:** "**prefix** / starts with / autocomplete / many words share beginnings / word dictionary" → a trie.
        Each step down costs O(1), so a lookup is O(length of the word), whatever the number of words.

**💡 The trick (trie):** Each node is a dictionary `children` (letter → node) plus a flag `end`. **insert** walks down,
        creating missing nodes, then sets `end = True` on the last node. **search** and **startsWith** walk the same
        way and return `False` if a letter is missing. The only difference: `search` also needs `end` on the last
        node. The classic bug: `search("app")` returning `True` after only `"apple"` was inserted, because `end` was
        not checked.

**⏱ Cost:** **O(L)** time per call for a word of length L · **O(total letters inserted)** space. A set of words gives O(L) `search` but `startsWith` would need a scan of all words.

In [ ]:
import random


class TrieNode:
    def __init__(self):
        self.children = {}                     # letter -> TrieNode
        self.end = False                       # does a word end here?


class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:        # no branch for this letter yet: grow one
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.end = True                        # tick: a word ends here

    def _walk(self, s):                        # the node at the end of s, or None if we fall off
        node = self.root
        for ch in s:
            node = node.children.get(ch)
            if node is None:
                return None
        return node

    def search(self, word):
        node = self._walk(word)
        return node is not None and node.end   # must be a whole word, not just a prefix

    def startsWith(self, prefix):              # LeetCode's exact method name
        return self._walk(prefix) is not None

    starts_with = startsWith

t = Trie()
assert t.search("") is False                   # nothing inserted yet
t.insert("apple")
assert t.search("apple") is True
assert t.search("app") is False                # only a prefix so far
assert t.startsWith("app") is True
t.insert("app")
assert t.search("app") is True
assert t.startsWith("b") is False and t.search("apples") is False
rng = random.Random(0)
for _ in range(100):                           # compare with a plain set of words
    words = {"".join(rng.choice("ab") for _ in range(rng.randint(1, 4))) for _ in range(5)}
    t = Trie()
    for w in words:
        t.insert(w)
    for q in ("".join(rng.choice("ab") for _ in range(rng.randint(1, 4))) for _ in range(10)):
        assert t.search(q) == (q in words)
        assert t.startsWith(q) == any(w.startswith(q) for w in words)
t = Trie()
t.insert("apple")
print(t.search("apple"), t.search("app"), t.startsWith("app"))

In [ ]:
def trie_panel(trie, hi=None, ehi=None, title="the trie (✓ = a word ends here)"):
    name, pos, edges, leaves = {}, {}, [], [0]
    def walk(node, prefix, depth):             # place leaves left to right, parents above their children
        name[prefix] = (prefix or "root") + ("✓" if node.end else "")
        xs = []
        for ch, child in sorted(node.children.items()):
            xs.append(walk(child, prefix + ch, depth + 1))
            edges.append((name[prefix], name[prefix + ch]))
        x = sum(xs) / len(xs) if xs else leaves[0]
        if not xs:
            leaves[0] += 1
        pos[name[prefix]] = (x, depth)
        return x
    walk(trie.root, "", 0)
    wide, deep = max(1, leaves[0] - 1), max(1, max(d for _, d in pos.values()))
    pos = {k: (x / wide if leaves[0] > 1 else 0.5, d / deep) for k, (x, d) in pos.items()}
    return graph(list(pos), edges, title, pos=pos,
                 hi={name[p]: r for p, r in (hi or {}).items() if p in name},
                 ehi={(name[a], name[b]): r for (a, b), r in (ehi or {}).items() if a in name and b in name})


def trie_movie(words, queries):
    m = Movie("Implement Trie", "Words that start the same share a branch")
    t = Trie()
    m.step("An empty trie is just the root. Each node below it will be named by the letters on the way down.",
           trie_panel(t))
    for word in words:
        node, prefix = t.root, ""
        for ch in word:
            grew = ch not in node.children
            if grew:
                node.children[ch] = TrieNode()
            node, prefix = node.children[ch], prefix + ch
            last = prefix == word
            if last:
                node.end = True
            path = {word[:k]: "win" for k in range(len(prefix))}
            m.step(f"insert(`{word}`): letter `{ch}` — " + ("no branch yet, **grow** a new node" if grew
                   else "the branch already exists, **share** it") +
                   (". Last letter: put a tick ✓ here." if last else "."),
                   trie_panel(t, {**path, prefix: "new" if grew else "look"},
                              {(word[:k], word[:k + 1]): "win" for k in range(len(prefix))}))
    results = []
    for kind, s in queries:
        node, k = t.root, 0
        while k < len(s) and s[k] in node.children:
            node, k = node.children[s[k]], k + 1
        fell_off = k < len(s)
        answer = (not fell_off) and (node.end if kind == "search" else True)
        results.append(answer)
        walked = {s[:j]: "win" for j in range(k)}
        walked[s[:k]] = "good" if answer else "bad"
        if fell_off:
            why = f"after `{s[:k] or 'root'}` there is no `{s[k]}` branch — we fall off"
        elif kind == "search":
            why = f"we reach `{s}`, and " + ("it has a tick ✓: a real word" if node.end else "it has **no** tick: only a prefix")
        else:
            why = f"we reach `{s}`, so some word starts with it"
        m.step(f"{kind}(`{s}`): walk down letter by letter; {why} → **{answer}**.",
               trie_panel(t, walked, {(s[:j], s[:j + 1]): "win" for j in range(k)}))
    ends = {w: "good" for w in words}
    m.step("Every call walked at most one node per letter — no matter how many words the trie holds. "
           "Green = the words with a tick ✓.", trie_panel(t, ends),
           hmap({f"{kind}({s})": str(r) for (kind, s), r in zip(queries, results)}, "answers",
                hi={f"{kind}({s})": ("good" if r else "bad") for (kind, s), r in zip(queries, results)}))
    return results, t, m

words = ["cat", "car", "dog"]
queries = [("search", "car"), ("search", "ca"), ("startsWith", "ca"), ("search", "cow")]
results, t_movie, m = trie_movie(words, queries)
real = Trie()
for w in words:
    real.insert(w)
assert results == [getattr(real, kind)(s) for kind, s in queries] == [True, False, True, False]
m.show()

---

## 80 · Network Delay Time · [LeetCode 743](https://leetcode.com/problems/network-delay-time/) · 🟡 Medium

**🧸 Like you're five:** You shout a secret in town 1. Each road takes a few minutes for the secret to travel, one way only.
        Always follow up the town the secret reaches **soonest**, because nothing can reach that town any sooner.
        When is the last town told?

**The question:** `times[i] = (u, v, w)`: a signal takes `w` time from node `u` to node `v` (directed, `w ≥ 0`). Nodes are `1..n`. Send a signal from `k`. Return the time for **all** nodes to receive it, or `-1` if some node never does.

| Input | Output |
|---|---|
| `times = [[2,1,1],[2,3,1],[3,4,1]], n = 4, k = 2` | `2` |
| `times = [[1,2,1]], n = 2, k = 1` | `1` |
| `times = [[1,2,1]], n = 2, k = 2` | `-1` |

**🔎 How to spot it:** "**shortest time / cheapest path** from one start in a graph with **weights ≥ 0**" → Dijkstra: a
        min-heap of `(time so far, node)`. (All weights equal? Plain BFS is enough. Negative weights? Bellman-Ford.)

**💡 The trick (dijkstra):** Push `(0, k)`. Pop the smallest time: if that node is already settled, it is an **old ticket** —
        skip it. Otherwise settle it (`dist[node] = time`) and push `(time + w, neighbour)` for each road out. The
        first pop of a node is its shortest time, because every other ticket in the heap is at least as big and
        weights are never negative. Answer: `max(dist.values())` if all n nodes are settled, else `-1`.

**⏱ Cost:** **O(E log E)** time (each road pushes at most one ticket) · **O(V + E)** space. Bellman-Ford is O(V · E).

In [ ]:
import heapq
import random
from collections import defaultdict


def network_delay_time(times, n, k):
    roads = defaultdict(list)
    for u, v, w in times:
        roads[u].append((v, w))
    dist = {}                                  # settled nodes: node -> shortest time
    heap = [(0, k)]                            # tickets: (time the signal arrives, node)
    while heap:
        t, u = heapq.heappop(heap)             # the soonest arrival of all
        if u in dist:
            continue                           # an old, slower ticket: already settled
        dist[u] = t
        for v, w in roads[u]:
            if v not in dist:
                heapq.heappush(heap, (t + w, v))
    return max(dist.values()) if len(dist) == n else -1


def delay_slow(times, n, k):                   # Bellman-Ford, only to check the answer
    INF = float("inf")
    d = [INF] * (n + 1)
    d[k] = 0
    for _ in range(n - 1):
        for u, v, w in times:
            d[v] = min(d[v], d[u] + w)
    best = max(d[1:])
    return -1 if best == INF else best

assert network_delay_time([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2) == 2
assert network_delay_time([[1, 2, 1]], 2, 1) == 1
assert network_delay_time([[1, 2, 1]], 2, 2) == -1     # the road only goes 1 -> 2
assert network_delay_time([], 1, 1) == 0               # only the start node
rng = random.Random(0)
for _ in range(300):
    n = rng.randint(1, 6)
    times = [[rng.randint(1, n), rng.randint(1, n), rng.randint(0, 9)] for _ in range(rng.randint(0, 12))]
    k = rng.randint(1, n)
    assert network_delay_time(times, n, k) == delay_slow(times, n, k)
print(network_delay_time([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2))

In [ ]:
def dijkstra_movie(times, n, k, pos):
    m = Movie("Network Delay Time", "Dijkstra: always settle the town the secret reaches soonest")
    roads = defaultdict(list)
    for u, v, w in times:
        roads[u].append((v, w))
    edges = [(u, v) for u, v, _ in times]
    weights = {(u, v): w for u, v, w in times}
    dist, heap, best_edge = {}, [(0, k)], {}

    def pic(caption, cur=None, pushed=(), stale=False):
        hi = {u: "done" for u in dist}
        if cur is not None:
            hi[cur] = "bad" if stale else "look"
        ehi = {e: "good" for e in best_edge.values()}
        ehi.update({(cur, v): "new" for v in pushed})
        m.step(caption, graph(range(1, n + 1), edges, "roads (minutes)", hi=hi, ehi=ehi, directed=True,
                              pos=pos, weights=weights),
               hmap(dist, "settled: town → time"),
               arr([f"({t},{u})" for t, u in sorted(heap)], "heap of tickets (time, town) — smallest time first",
                   hi={0: "look"} if heap else None, index=False))

    pic(f"Shout the secret in town {k} at time 0: the heap holds one ticket `(0,{k})` = (time, town).")
    came_from = {}
    while heap:
        t, u = heapq.heappop(heap)
        if u in dist:
            pic(f"Pop `({t},{u})`, but town {u} was already settled at time {dist[u]}. An **old, slower ticket**: skip.",
                u, stale=True)
            continue
        dist[u] = t
        if u in came_from:
            best_edge[u] = came_from[u][t]
        pushed = []
        for v, w in roads[u]:
            if v not in dist:
                heapq.heappush(heap, (t + w, v))
                came_from.setdefault(v, {})[t + w] = (u, v)
                pushed.append(v)
        news = ", ".join(f"`({t + w},{v})`" for v, w in roads[u] if v in pushed)
        pic(f"Pop the smallest ticket `({t},{u})`: town {u} hears it at **{t}** — nothing can be sooner. "
            + (f"Push {news}." if pushed else "No new roads out."), u, pushed)
    answer = max(dist.values()) if len(dist) == n else -1
    m.step(f"Every town is settled. The last one hears the secret at **{answer}** minutes. Green roads are "
           f"the fastest routes.", graph(range(1, n + 1), edges, "roads (minutes)",
                                         hi={u: "good" for u in dist}, ehi={e: "good" for e in best_edge.values()},
                                         directed=True, pos=pos, weights=weights),
           hmap(dist, "settled: town → time", hi={u: "good" for u in dist if dist[u] == answer}))
    return answer, m

times = [[1, 2, 4], [1, 3, 1], [3, 2, 2], [2, 4, 1], [3, 4, 5], [4, 5, 3]]
pos = {1: (0, 0.5), 2: (0.35, 0), 3: (0.35, 1), 4: (0.7, 0.5), 5: (1, 0.5)}
answer, m = dijkstra_movie(times, 5, 1, pos)
assert answer == network_delay_time(times, 5, 1) == 7
m.show()

---

## Quiz yourself: which trick?

Eight LeetCode questions that are **not** in the top 100. Decide the trick before you open the answer.

1. **Last Stone Weight** (1046, Easy) — smash the two heaviest stones together, again and again; what is left?
   <details><summary>which trick?</summary><b>Max-heap</b> (push negatives). You need "the two biggest" over and over while the pile changes.</details>
2. **Kth Largest Element in a Stream** (703, Easy) — numbers keep arriving; after each, report the k-th largest.
   <details><summary>which trick?</summary><b>Min-heap of size k</b> (problem 61), kept between calls: its top is always the answer.</details>
3. **Combination Sum II** (40, Medium) — like problem 66, but each number may be used **once** and the input has duplicates.
   <details><summary>which trick?</summary><b>Backtracking</b> with <code>go(i + 1)</code> and, after sorting, skip <code>nums[i] == nums[i - 1]</code> at the same level so equal piles are not made twice.</details>
4. **Palindrome Partitioning** (131, Medium) — cut a string into pieces that are all palindromes; list every way.
   <details><summary>which trick?</summary><b>Backtracking</b>: choose the next piece <code>s[start:end]</code> only if it is a palindrome, recurse on the rest, undo.</details>
5. **01 Matrix** (542, Medium) — for every cell, the distance to the nearest 0.
   <details><summary>which trick?</summary><b>Multi-source BFS</b> (problem 73): queue every 0 at once; the rings give the distances.</details>
6. **Surrounded Regions** (130, Medium) — flip every `O` region that does not touch the border.
   <details><summary>which trick?</summary><b>Search backwards from the border</b> (problem 77): flood from the edge <code>O</code>s to mark the safe ones, then flip the rest.</details>
7. **Redundant Connection** (684, Medium) — a tree plus one extra edge; which edge makes the cycle?
   <details><summary>which trick?</summary><b>Union-find</b> (problem 78): add edges one by one; the first edge whose two ends already have the same boss is the answer.</details>
8. **Design Add and Search Words Data Structure** (211, Medium) — like a trie, but a search word may contain `.` meaning "any letter".
   <details><summary>which trick?</summary><b>Trie + DFS</b> (problem 79): at a <code>.</code>, try every child branch; that is backtracking inside the trie.</details>

## Pattern cheat sheet

| If the question says… | think… | example here |
|---|---|---|
| "k-th largest", "top k", "k biggest" | min-heap that never holds more than k | 61 |
| "k closest", "k smallest" | max-heap of size k (store negatives) | 62 |
| "the same task must wait n turns", "schedule to finish soonest" | greedy with a max-heap + a waiting queue | 63 |
| "all subsets", "all combinations" | backtracking; start the loop at `start` | 64, 66, 67 |
| "all orderings / arrangements" | backtracking with a `used` list | 65 |
| "generate all **valid** …" | backtracking that only takes legal steps (counters) | 68 |
| "path in a grid, each cell once" | DFS + mark / unmark | 69 |
| "count islands / regions / groups in a grid" | flood fill from every unvisited cell | 70, 71, 72 |
| "spreads every minute", "fewest steps", several starts | multi-source BFS, ring by ring | 73 |
| "copy a graph / list with cycles" | dictionary old → new while walking | 74 |
| "prerequisites", "must come before", "is it possible to finish" | topological sort (Kahn); leftovers = cycle | 75, 76 |
| "which cells can reach the edge / both sides" | search **backwards** from the targets | 77 |
| "how many groups", "are they connected", links added one by one | union-find | 78 |
| "prefix", "starts with", "autocomplete" | trie | 79 |
| "shortest time / cheapest path", weights ≥ 0 | Dijkstra: heap of (distance, node) | 80 |

## Say it in an interview

- **Clarify first.** Can values repeat? Is the graph directed? Can weights be negative? Is the grid allowed to be
  changed in place? Any order for the answer? (Problems 62, 64–68 and 76 all accept any order — say so.)
- **Brute force, then the trick, by name.** "Sorting is O(n log n); a heap of size k gives O(n log k)." "I'll
  backtrack: choose, explore, undo." "This is BFS because every step costs the same and I want the fewest steps."
- **Backtracking:** say the three moves out loud, and the complexity is the *size of the output* (2ⁿ subsets,
  n! orderings). The traps: saving `path` instead of `path[:]`, and forgetting to undo (pop / unmark / `used[i] =
  False`).
- **Grids and graphs:** mark a cell as visited **when you queue it**, not when you pop it. Prefer an explicit
  queue or stack over recursion on big grids (Python's recursion limit is about 1000). Check the 4 neighbours
  with one small loop and one bounds check.
- **Direction and edges:** `[a, b]` in Course Schedule means **b → a**. Dijkstra needs weights ≥ 0 and must skip
  old heap tickets. Multi-source BFS queues *all* sources before the first minute.
- **Test out loud:** empty input, one element, all water / all land, a cycle, an unreachable node (`-1`), and
  the "nothing to do" case (Flood Fill with the same colour).

## Next

- ⬅ Previous: [17 · Top 100, part 3 — linked lists and trees](17_top100_part3_linked_lists_and_trees.ipynb)
- ➡ Next: [19 · Top 100, part 5 — dynamic programming, greedy, intervals and matrix](19_top100_part5_dp_greedy_intervals_matrix.ipynb)
- Go deeper: [08 · Recursion and backtracking](08_recursion_and_backtracking.ipynb) ·
  [10 · Heaps and top-k](10_heaps_and_top_k.ipynb) ·
  [11 · Graphs: BFS, DFS, topo sort, Dijkstra](11_graphs_bfs_dfs_topo_dijkstra.ipynb) ·
  [14 · Tries, union-find and LRU cache](14_tries_union_find_lru_cache.ipynb)